# MovieLens 32M

Author: Michael Fitiavana Randriamifidy
Created: Nov 2025 
Last Updated: 24 Nov 2025

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from typing import Tuple, Union, Optional
import numpy.linalg as la
from tqdm import tqdm
from collections.abc import Callable
from numba import njit, prange, types
from numba.typed import List
from tqdm import tqdm
import time
from pathlib import Path
import os, sys
import csv
import pprint
import pandas as pd
import subprocess
import yaml
import uuid
from datetime import datetime, timedelta
from joblib import dump, load
from typing import Any, Tuple
import re
from numba.typed import List
import numba
from numba import types, typeof
from numba import  int64, float64
import csv
import pprint
import pandas as pd
import subprocess
import textwrap
import warnings
import umap

In [ ]:
from datetime import datetime, timedelta
def show_execution_duration(info:str="", return_time=False):
    if len(info):
        print(f"{info}")
    def inner_fun(f:Callable)-> Callable:
        def wrapper(*args, **kwargs):
            print(f"Execution of:  {f.__name__}")
            start = datetime.now()
            print(f"Start at: {start.strftime('%Y-%m-%d_%H-%M-%S')}")
            result = f(*args, **kwargs)
            end = datetime.now()
            print(f"Finished at: {end.strftime('%Y-%m-%d_%H-%M-%S')}")
            duration = (end - start)
            print(f"Execution in: {duration}")
            if return_time:
                return result, start, duration
            else:
                return result
        return wrapper
    return inner_fun

In [ ]:
def dump_objects(
    mobject: Any,
    path: str = "models/data.joblib",
    compress: int = 3,
    timestamp: bool = False,
)-> str:
    # Add timestamp if requested
    if timestamp:
        base, ext = os.path.splitext(path)
        current_time = datetime.now().strftime("%Y-%m-%d_%H-%M")
        path = f"{base}_{current_time}{ext}"

    os.makedirs(os.path.dirname(path), exist_ok=True)
    dump(mobject, path, compress=compress)
    print(f"Objects successfully saved to '{path}'")
    return path


def load_objects(path: str = "models/data.joblib") -> Tuple[Any, ...]:
    objs = load(path)
    print(f"Objects successfully loaded from '{path}'")
    return objs

def save_plot(
    path,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    overwrite=False,
    dpi=300,
):
    # store original LaTeX setting
    old_usetex = plt.rcParams.get("text.usetex", False)

    # try per-plot LaTeX rendering with fallback
    if use_tex:
        try:
            plt.rcParams["text.usetex"] = True
            with warnings.catch_warnings():
                warnings.simplefilter("error")
                (fig or plt.gcf()).canvas.draw()
        except Exception:
            plt.rcParams["text.usetex"] = False

    # suppress Matplotlib warnings
    warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

    # create parent folder if missing
    dirname = os.path.dirname(path)
    if dirname:
        os.makedirs(dirname, exist_ok=True)

    # use current figure if none provided
    if fig is None:
        fig = plt.gcf()

    # strip file extension from path
    base, _ = os.path.splitext(path)

    # expand "all" keyword or convert single format to list
    if formats == "all":
        formats = ["pdf", "png", "svg"]
    elif isinstance(formats, str):
        formats = [formats]

    # build timestamp string if enabled
    timestamp = time.strftime("%Y%m%d_%H%M%S") if timestamps else ""

    saved = []

    # iterate over formats and save each
    for fmt in formats:
        # construct final filename with optional timestamp
        final = f"{base}_{timestamp}.{fmt}" if timestamp else f"{base}.{fmt}"

        # auto-increment filename if overwrite=False and file exists
        if not overwrite and os.path.exists(final):
            stem, ext = os.path.splitext(final)
            i = 1
            alt = f"{stem}_{i}{ext}"
            while os.path.exists(alt):
                i += 1
                alt = f"{stem}_{i}{ext}"
            final = alt

        # save figure
        fig.savefig(final, dpi=dpi, bbox_inches="tight")
        saved.append(final)

    # restore original LaTeX setting
    plt.rcParams["text.usetex"] = old_usetex

    # return list of saved file paths
    return saved



def search_regex(lst, pattern, flags=0):
    regex = re.compile(pattern, flags)
    return [i for i, s in enumerate(lst) if regex.search(s)]

def get_paths(folder: str | Path, pattern: str) -> list[Path]:
    """
    Return list of file paths in `folder` matching the given pattern.
    Uses recursive search.
    """
    folder = Path(folder)
    regex = re.compile(pattern)
    return [p for p in folder.iterdir() if regex.match(p.name)]

In [ ]:
import yaml
import uuid
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np

def ensure_float_list(v):
    """Convert scalars, numpy scalars, arrays → clean Python float list."""
    if isinstance(v, (int, float, np.generic)):
        return [float(v)]
    if isinstance(v, np.ndarray):
        return [float(x) for x in v.tolist()]
    if isinstance(v, list):
        return [float(x) for x in v]
    return [float(v)]


def sanitize_value(v):
    """Convert types YAML can't handle → simple strings/floats."""
    if isinstance(v, datetime):
        return v.isoformat(timespec="seconds")
    if isinstance(v, timedelta):
        return str(v)
    if isinstance(v, np.generic):
        return v.item()
    if isinstance(v, np.ndarray):
        return [sanitize_value(x) for x in v.tolist()]
    return v


def sanitize_entry(entry: dict) -> dict:
    """Apply sanitize_value recursively."""
    out = {}
    for k, v in entry.items():
        if isinstance(v, dict):
            out[k] = sanitize_entry(v)
        elif isinstance(v, list):
            out[k] = [sanitize_value(x) for x in v]
        else:
            out[k] = sanitize_value(v)
    return out


def log_als_run_to_yaml(log_path: Path, arg: dict, suffix:str=""):
    """
    Append a new ALS run to a global YAML log.
    Guarantees:
      - rmse/loss are always float lists (last 10)
      - timestamps sanitised
      - YAML clean and safe
      - model_path removed
    """
    entry = arg.copy()

    # metadata
    entry["run_id"] = str(uuid.uuid4())
    entry["timestamp"] = datetime.now()

    # enforce rmse/loss as float lists
    for key in ["train_rmse", "train_loss", "test_rmse", "test_loss"]:
        if key in entry:
            entry[key] = ensure_float_list(entry[key])[-10:]

    # sanitize for yaml
    entry = sanitize_entry(entry)

    # load existing log
    if os.path.exists(log_path):
        with open(log_path, "r", encoding="utf-8") as f:
            try:
                log_data = yaml.safe_load(f) or []
            except yaml.YAMLError:
                log_data = []
    else:
        log_data = []

    # append
    log_data.append(entry)

    # save
    base, ext = os.path.splitext(log_path)
    path = f"{base}_{suffix}{ext}"


    with open(log_path, "w", encoding="utf-8") as f:
        yaml.dump(log_data, f, sort_keys=True, allow_unicode=True)

    return entry["run_id"], log_path


# Practical 1

In [ ]:
PROJECT_PATH = Path("/content/drive/MyDrive/Labs/ml_scale/recommender")
data_folder = PROJECT_PATH / "data"
plot_folder = PROJECT_PATH / "plots"
model_folder = PROJECT_PATH / "models"
extracted_data_folder = Path("./data")

PROJECT_PATH.mkdir(exist_ok=True)
extracted_data_folder.mkdir(exist_ok=True)
# os.chdir(PROJECT_PATH)
# print(f"Working directory: {os.getcwd()}")

# if PROJECT_PATH not in sys.path:
#     sys.path.append(PROJECT_PATH)
#     print(f"Added to sys.path: {PROJECT_PATH}")
# %load_ext autoreload
# %autoreload 2
!pwd
!ls $PROJECT_PATH -R | head -20

In [ ]:
# !curl -O https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!curl -O  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip.md5

# !curl -O  https://files.grouplens.org/datasets/movielens/ml-32m.zip
!curl -O  https://files.grouplens.org/datasets/movielens/ml-32m.zip.md5

# !curl -O  https://files.grouplens.org/datasets/movielens/ml-32m.zip
!curl -O  https://files.grouplens.org/datasets/movielens/ml-32m.zip.md5

In [ ]:
# #  Extract data
!unzip "{data_folder}/ml-latest-small.zip" -d $extracted_data_folder
# # !unzip "{data_folder}/ml-25m.zip" -d $extracted_data_folder
!unzip "{data_folder}/ml-32m.zip" -d $extracted_data_folder

In [ ]:
# !head -n 10 {extracted_data_folder}/ml-25m/ratings.csv
# print()
# !head -n 10 {extracted_data_folder}/ml-25m/ratings.csv

## Data Structure indexes (a) by user and (b) by movie


In [ ]:
# ml-latest-small or ml-25m or ml-32m
# data_size_str = "ml-latest-small"
data_size_str = "ml-32m"
ratings_path = extracted_data_folder / data_size_str / "ratings.csv"
movies_path = extracted_data_folder / data_size_str / "movies.csv"

## CSR

In [ ]:
def build_csr(sparse_matrix):
    """
    Convert list[list[(idx, value)]] into CSR arrays.
    """
    row_offset = np.zeros(len(sparse_matrix) + 1, dtype=np.int64)
    col_indices = []
    values = []

    for i, row in enumerate(sparse_matrix):
        row_offset[i + 1] = row_offset[i] + len(row)
        for col, val in row:
            col_indices.append(col)
            values.append(val)

    return (
        row_offset,
        np.array(col_indices, dtype=np.int32),
        np.array(values, dtype=np.float32),
    )

def get_nelements_by_row_csr(matrix_csr: tuple[np.ndarray, np.ndarray, np.ndarray]):
    row_offset, col_indices, values = matrix_csr
    return np.array(row_offset)[1:] - np.array(row_offset)[:-1]

def get_degree_frequency_csr(sparse_matrix_csr: tuple[np.ndarray]):
    """
    Compute (degree, frequency) from CSR matrix.
    degree = number of non-zeros in a row.
    """
    degrees = get_nelements_by_row_csr(sparse_matrix_csr)
    x_degree, y_frequency = np.unique(degrees, return_counts=True)
    return x_degree, y_frequency
@njit
def get_row_from_csr(matrix_csr, row_idx):
    row_offset, col_indices, values = matrix_csr
    start = row_offset[row_idx]
    end = row_offset[row_idx + 1]
    return col_indices[start:end], values[start:end]

@njit
def get_nrows_csr(matrix_csr):
    row_offset, _, _ = matrix_csr
    return len(row_offset) - 1

In [ ]:
def load_movie_id_idx_mapping(
    ratings_path: str
    )->tuple[dict, np.ndarray]:
    # Maps
    movie_id_to_idx = {}
    movie_idx_to_id = []

    total = int(subprocess.check_output(["wc", "-l", ratings_path]).split()[0])
    with open(ratings_path, "r", encoding="UTF-8") as f:

        next(f)

        for line in tqdm(f,total=total, unit=" lines"):
            userId, movieId, rating, _ = line.split(",")
            movieId = int(movieId)
            # Map movie ID
            if movieId not in movie_id_to_idx:
                midx = len(movie_id_to_idx)
                movie_id_to_idx[movieId] = midx
                movie_idx_to_id.append(movieId)

    return (
        movie_id_to_idx,
        movie_idx_to_id
    )

In [ ]:
@show_execution_duration()
def load_ratings_data_csr(ratings_path: str
                    )->tuple[dict, np.ndarray,
                    dict, np.ndarray,
                    tuple[np.ndarray, np.ndarray, np.ndarray],
                    tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Parse ratings file and produce efficient array-based CSR-style structures.
    Numba-ready
    """
    # Maps

    user_id_to_idx = {}
    user_idx_to_id = []
    movie_id_to_idx = {}
    movie_idx_to_id = []

    # Temporary buckets for building CSR
    ratings_by_user = []     # each element: list[(movie_idx, rating)]
    ratings_by_movie = []
    total = int(subprocess.check_output(["wc", "-l", ratings_path]).split()[0])
    with open(ratings_path, "r", encoding="UTF-8") as f:

        next(f)

        for line in tqdm(f,total=total, unit=" lines"):
            userId, movieId, rating, _ = line.split(",")
            userId = int(userId)
            movieId = int(movieId)
            rating = float(rating)

            # Map user ID
            if userId not in user_id_to_idx:
                uidx = len(user_id_to_idx)
                user_id_to_idx[userId] = uidx
                user_idx_to_id.append(userId)
                ratings_by_user.append([])
            uidx = user_id_to_idx[userId]

            # Map movie ID
            if movieId not in movie_id_to_idx:
                midx = len(movie_id_to_idx)
                movie_id_to_idx[movieId] = midx
                movie_idx_to_id.append(movieId)
                ratings_by_movie.append([])
            midx = movie_id_to_idx[movieId]

            ratings_by_user[uidx].append((midx, rating))
            ratings_by_movie[midx].append((uidx, rating))

    # Convert to CSR arrays

    ratings_by_user_csr = build_csr(ratings_by_user)
    ratings_by_movie_csr = build_csr(ratings_by_movie)

    return (
        user_id_to_idx,
        user_idx_to_id,
        movie_id_to_idx,
        movie_idx_to_id,
        ratings_by_user_csr,
        ratings_by_movie_csr,
    )


In [ ]:
def load_ratings_and_features_data_csr(
    ratings_path: str,
    data_movies_path:str,
    split: float = 0.9,
    seed=42,
    load_movie_data_only=False):
    """
    Parse ratings file and produce efficient array-based CSR-style structures.
    Numba-ready.
    """
    np.random.seed(seed)
    # Maps
    user_id_to_idx = {}
    movie_id_to_idx = []

    user_id_to_idx = {}
    user_idx_to_id = []
    movie_id_to_idx = {}
    movie_idx_to_id = []

    # Temporary buckets for building CSR
    ratings_by_user = []
    ratings_by_movie = []

    total = int(subprocess.check_output(["wc", "-l", ratings_path]).split()[0])
    with open(ratings_path, "r", encoding="UTF-8") as f:

        next(f)

        for line in tqdm(f,total=total, unit=" lines",  desc="Loading ratings"):
            userId, movieId, rating, _ = line.split(",")
            userId = int(userId)
            movieId = int(movieId)
            rating = float(rating)

            # Map user ID
            if userId not in user_id_to_idx:
                uidx = len(user_id_to_idx)
                user_id_to_idx[userId] = uidx
                user_idx_to_id.append(userId)
                ratings_by_user.append([])

            uidx = user_id_to_idx[userId]

            # Map movie ID
            if movieId not in movie_id_to_idx:
                midx = len(movie_id_to_idx)
                movie_id_to_idx[movieId] = midx
                movie_idx_to_id.append(movieId)
                ratings_by_movie.append([])

            midx = movie_id_to_idx[movieId]


            ratings_by_user[uidx].append((midx, rating))
            ratings_by_movie[midx].append((uidx, rating))

        feature_id_to_idx = {}
        feature_idx_to_id = []

        movies_title = [ [] for _ in range(len(movie_idx_to_id))]
        movies_by_feature = []
        features_by_movie = [ [] for _ in range(len(movie_idx_to_id))]

        total = int(subprocess.check_output(["wc", "-l", data_movies_path]).split()[0]) - 1
        with open(data_movies_path, "r", encoding="UTF-8") as f:
            next(f)

            for line in tqdm(f,total=total, unit=" lines", desc="Loading features"):
                movieId, title, features_str =  next(csv.reader([line]))
                movieId = int(movieId)
                featuresIds = features_str.split("|")
                if movieId in movie_id_to_idx:
                    midx = movie_id_to_idx[movieId]
                    movies_title[midx] = title
                    # Map Features ID
                    for featureId in featuresIds:
                        if featureId not in feature_id_to_idx:
                            fidx = len(feature_id_to_idx)
                            feature_id_to_idx[featureId] = fidx
                            feature_idx_to_id.append(featureId)
                            movies_by_feature.append([])
                        fidx = feature_id_to_idx[featureId]
                        movies_by_feature[fidx].append((midx, 1))
                        features_by_movie[midx].append((fidx, 1))

    # Convert to CSR arrays

    ratings_by_user_csr = build_csr(ratings_by_user)
    ratings_by_movie_csr = build_csr(ratings_by_movie)

    movies_by_feature_csr = build_csr(movies_by_feature)
    features_by_movie_csr = build_csr(features_by_movie)

    return (
        user_id_to_idx,
        user_idx_to_id,
        movie_id_to_idx,
        movie_idx_to_id,
        feature_idx_to_id,
        feature_id_to_idx,
        ratings_by_user_csr,
        ratings_by_movie_csr,
        features_by_movie_csr,
        movies_by_feature_csr,
        movies_title,
    )

def sparsity_of_matrix_csr(sparse_matrix_csr, shape):
    M, N = shape
    nnz = get_nelements_by_row_csr(sparse_matrix_csr).sum()
    sparsity =  nnz / (N * M)
    return nnz, sparsity

In [ ]:
(
    user_id_to_idx, user_idx_to_id, movie_id_to_idx, movie_idx_to_id,
    feature_idx_to_id, feature_id_to_idx,
    ratings_by_user_csr, ratings_by_movie_csr,
    features_by_movie_csr, movies_by_feature_csr, movies_title
)= load_ratings_and_features_data_csr(ratings_path, movies_path)
assert get_nrows_csr(ratings_by_user_csr) == get_nrows_csr(ratings_by_user_csr)

In [ ]:
# del ratings_by_user_csr
# del ratings_by_movie_csr

In [ ]:
# del features_by_movie_csr
# del movies_by_feature_csr

### Rating Matrix sparsity

In [ ]:
N = get_nrows_csr(ratings_by_movie_csr)
M = get_nrows_csr(ratings_by_user_csr)
nratings, ratings_sparsity = sparsity_of_matrix_csr(ratings_by_user_csr, (M, N))
print(f"Rating Matrix sparsity \t:{ratings_sparsity:.5f}\n" \
f"Number of ratings \t: {nratings}")

### Plots

In [ ]:
def plot_ratings_csr_publication(
    ratings_csr,
    label=None,
    title=None,
    log_plot=True,
    bw_plot=True,
    color=None,
    figsize=(7,5),
    fig_ax=None,
    hist=False,
    bins=1000,
):
    """
    Plot rating-degree distributions for CSR matrices.
    hist=False  → scatter
    hist=True   → histogram with custom bins
    """

    x_degree, y_frequency = get_degree_frequency_csr(ratings_csr)

    # FIGURE CREATION (non-bar modes)
    if fig_ax:
        fig, ax = fig_ax
    else:
        fig, ax = plt.subplots(figsize=figsize)

    if bw_plot:
        color = "black"

    # HISTOGRAM MODE
    if hist is True:
        expanded = np.repeat(x_degree, y_frequency)
        if bw_plot:
            color = 'lightgray' if bw_plot else None
        ax.hist(
            expanded,
            bins=bins,
            color=color,
            alpha=0.7,
            edgecolor="black",
            label=label,
        )

        if log_plot:
            ax.set_xscale("log")

        ax.set_xlabel("number of Ratings")
        ax.set_ylabel("Count")
        ax.set_title(title)
        ax.grid(False)
        plt.tight_layout()
        return fig, ax

    # SCATTER MODE
    ax.scatter(x_degree, y_frequency, s=2, alpha=0.7, color=color, label=label)

    if log_plot:
        ax.set_xscale("log")
        ax.set_yscale("log")

    ax.set_xlabel("Degree")
    ax.set_ylabel("Frequency")
    ax.set_title(title)
    ax.grid(False)
    plt.tight_layout()

    return fig, ax




# Predefined markers and line styles for BW plots
BW_MARKERS = ['o', 's', '^', 'D', 'x', '*', 'v', 'P']
BW_LINESTYLES = ['-', '--', '-.', ':']

def plot_bw_scatter_lines(data_list, labels=None, log_scale=False, title="BW Plot",figsize=(7,5), fig_ax=None):
    """
    Plot multiple series in black-and-white using different markers and line styles.
    """
    if fig_ax:
        fig, ax = fig_ax
    else:
        fig, ax = plt.subplots(figsize=figsize)

    n_series = len(data_list)
    for i, (x, y) in enumerate(data_list):
        marker = BW_MARKERS[i % len(BW_MARKERS)]
        linestyle = BW_LINESTYLES[i % len(BW_LINESTYLES)]
        label = labels[i] if labels else None

        ax.plot(x, y, marker=marker, linestyle=linestyle, color='black', alpha=0.8, label=label)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_title(title)
    ax.grid(False)
# Information
    if log_scale:
        ax.set_xscale('log')
        ax.set_yscale('log')

    if labels:
        ax.legend()

    plt.tight_layout()
    return fig, ax

In [ ]:
def power_law_fit(x:np.ndarray, y:np.ndarray):
    logx = np.log(x)
    logy = np.log(y)
    b, loga = np.polyfit(logx, logy, 1)   # slope, intercept
    a = np.exp(loga)
    return a, b

def power_law_predict(x:np.ndarray, a:float, b:float) ->np.ndarray:
    return a * x**b

def get_power_law_from_csr(ratings_csr):
    x_degree, y_frequency = get_degree_frequency_csr(ratings_csr)
    a, gamma  = power_law_fit(x_degree, y_frequency)
    return a, gamma

In [ ]:
# power law Users
type_vector = "Users"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "
plot_folder = Path(".") / "plots"
filename = f"power-law-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_user_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C0',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

a, gamma = get_power_law_from_csr(ratings_by_user_csr)
print(f"{type_vector}: p(d) proportional to  d^({gamma:.2f})")

In [ ]:
# power law Movies
type_vector = "Movies"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "

plot_folder = Path(".") / "plots"
filename = f"power-law-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_movie_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C1',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

a, gamma = get_power_law_from_csr(ratings_by_movie_csr)
print(f"{type_vector}: p(d) proportional to  d^({gamma:.2f})")

In [ ]:
type_vector = "Users"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "

plot_folder = Path(".") / "plots"
filename = f"ratings-distribution-filtered-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)

# bins= [0, 10**2, 200 , 300, 400, 500, 600, 700,  800, 900, 10**3, 10**4, np.inf]
bins = 10**3

plot_ratings_csr_publication(ratings_by_user_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=True,
                            color='C0',
                            hist=True,
                            bins= bins
                            )
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

In [ ]:
type_vector = "Movies"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "
bins = 10**4

plot_folder = Path(".") / "plots"
filename = f"ratings-distribution-filtered-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)

plot_ratings_csr_publication(ratings_by_movie_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=True,
                            color='C0',
                            hist=True,
                            bins= bins
                            )
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

In [ ]:
def get_values_per_bin(values, counts, bins):
    """
    Aggregate counts into bins using '>' for the last bin.
    """
    values = np.array(values)
    counts = np.array(counts)

    bin_indices = np.digitize(values, bins, right=False)

    bin_labels = []
    bin_totals = []

    for i in range(1, len(bins)):
        lo = bins[i-1]
        hi = bins[i]

        mask = bin_indices == i
        total = int(np.sum(counts[mask]))

        if np.isinf(hi):
            label = f">{lo}"
        else:
            label = f"{lo}–{hi}"

        bin_labels.append(label)
        bin_totals.append(total)

    return bin_labels, bin_totals


# Movies Features

In [ ]:
def load_data_movie_feature_csr(
        data_movies_path: str
        )->tuple[dict, np.ndarray,
        dict, np.ndarray,
        np.ndarray,
        tuple[np.ndarray, np.ndarray, np.ndarray],
        tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Parse Movie Genres into CSR data structure
    """
    # Maps
    data_movie_id_to_idx = {}
    data_movie_idx_to_id = []

    data_feature_id_to_idx = {}
    data_feature_idx_to_id = []

    # Temporary buckets for building CSR
    data_movies_title = []
    movies_by_feature = []
    features_by_movie = []
    np.array([1])
    total = int(subprocess.check_output(["wc", "-l", data_movies_path]).split()[0]) - 1
    with open(data_movies_path, "r", encoding="UTF-8") as f:

        next(f)

        for line in tqdm(f,total=total, unit=" lines"):
            movieId, title, features_str =  next(csv.reader([line]))
            movieId = int(movieId)
            featuresIds = features_str.split("|")

            # Map movie ID
            if movieId not in data_movie_id_to_idx:
                midx = len(data_movie_id_to_idx)
                data_movie_id_to_idx[movieId] = midx
                data_movie_idx_to_id.append(movieId)
                data_movies_title.append(title)
                features_by_movie.append([])
            midx = data_movie_id_to_idx[movieId]

            # Map Features ID
            for featureId in featuresIds:
                if featureId not in data_feature_id_to_idx:
                    fidx = len(data_feature_id_to_idx)
                    data_feature_id_to_idx[featureId] = fidx
                    data_feature_idx_to_id.append(featureId)
                    movies_by_feature.append([])
                fidx = data_feature_id_to_idx[featureId]
                movies_by_feature[fidx].append((midx, 1))
                features_by_movie[midx].append((fidx, 1))

    # Convert to CSR arrays
    movies_by_feature_csr = build_csr(movies_by_feature)
    features_by_movie_csr = build_csr(features_by_movie)

    return (
    data_movie_id_to_idx,
    data_movie_idx_to_id,
    data_feature_id_to_idx,
    data_feature_idx_to_id,
    data_movies_title,
    movies_by_feature_csr,
    features_by_movie_csr,
    )

def get_idx_from_id(ids:Union[int, list[int]], id_to_idx_map:list) -> Union[int, list[int]]:

    if isinstance(ids,(int, np.integer)):
        return id_to_idx_map[ids]
    indices = [id_to_idx_map[id] for id in ids]
    return indices

def get_id_from_idx(indices:Union[int, list[int]], idx_to_id_map:list) -> Union[int, list[int]]:
    ids = np.array(idx_to_id_map)[indices]
    if isinstance(indices,(int, np.integer)):
        return ids.item()
    return ids

def get_id_from_idx(indices:Union[int, list[int]], idx_to_id_map:list) -> Union[int, list[int]]:
    if isinstance(indices, (int, np.integer)):
        return idx_to_id_map[indices]
    else:
        return [idx_to_id_map[i] for i in indices]

def get_movie_title_from_id(ids:Union[int, list[int]], movie_id_to_idx:dict, data_movies_title:list[str]) -> Union[str, list[str]]:
    if isinstance(ids, (int, np.integer)):
        idx = movie_id_to_idx[ids]
        return data_movies_title[idx]
    else:
        indices = [movie_id_to_idx[i] for i in ids]
        return [data_movies_title[i] for i in indices]


def get_features_id_from_movie_id(
        id:int,
        movie_id_to_idx:dict,
        feature_idx_to_id:list[int],
        features_by_movie_csr:Tuple[np.ndarray, np.ndarray, np.ndarray]
        ) -> list[str]:
        movie_idx = get_idx_from_id(id, movie_id_to_idx)
        features_indices, _ = get_row_from_csr(features_by_movie_csr, movie_idx)
        features =  get_id_from_idx(features_indices, feature_idx_to_id)
        return features

def get_movies_id_from_feature_id(
        id:int,
        feature_id_to_idx:dict,
        movie_idx_to_id:list[int],
        movies_by_feature_csr:Tuple[np.ndarray, np.ndarray, np.ndarray]
        ) -> list[str]:
        movie_idx = get_idx_from_id(id, feature_id_to_idx)
        movies_indices, _ = get_row_from_csr(movies_by_feature_csr, movie_idx)
        movies =  get_id_from_idx(movies_indices, movie_idx_to_id)
        return movies

def find_movie_id_by_title(title:str,
                        data_movies_title:list[str],
                        movie_idx_to_id:list[int],
                        data_movie_id_to_idx:dict,
                        )->tuple[list[int],list[str]]:
    title_pattern = title
    movies_idx = search_regex(data_movies_title, title_pattern)
    npertinent = len(movies_idx)
    movies_ids = []
    movies_title = []
    for i, movie_idx in enumerate(movies_idx):
        movie_id = get_id_from_idx(movie_idx, movie_idx_to_id)
        movie_title= get_movie_title_from_id(
        movie_id, data_movie_id_to_idx, data_movies_title)
        movies_ids.append(movie_id)
        movies_title.append(movie_title)
    return movies_ids, movies_title

def get_movie_info_from_id(movies_ids:list[int],
                           data_movie_id_to_idx:dict,
                           data_feature_idx_to_id:list[str],
                           data_movies_title:list[str],
                           features_by_movie_csr:Tuple[np.ndarray, np.ndarray, np.ndarray]
                           )->list[dict]:
    npertinent = len(movies_ids)
    results = [{} for _ in range(npertinent)]
    for i, movie_id in enumerate(movies_ids):
        movie_title = get_movie_title_from_id(movie_id, data_movie_id_to_idx, data_movies_title)
        features = get_features_id_from_movie_id(movie_id, data_movie_id_to_idx, data_feature_idx_to_id,
                                                 features_by_movie_csr)
        results[i] = {"id": movie_id, "title": str(movie_title), "features":features}
    return results

In [ ]:
(
    data_movie_id_to_idx,
    data_movie_idx_to_id,
    data_feature_id_to_idx,
    data_feature_idx_to_id,
    data_movies_title,
    movies_by_feature_csr,
    features_by_movie_csr,
    ) = load_data_movie_feature_csr(movies_path)

movie_id_to_idx, movie_idx_to_id = load_movie_id_idx_mapping(ratings_path)

Search Movie by title

In [ ]:
title = "Lord"
ids, titles = find_movie_id_by_title(title, movies_title, movie_idx_to_id, movie_id_to_idx)
results = get_movie_info_from_id(
    ids,
    movie_id_to_idx,
    feature_idx_to_id,
    movies_title,
    features_by_movie_csr
    )
pd.DataFrame(results)

## EDA

In [ ]:
def compute_csr_statistics(features_by_movie_csr, movies_by_feature_csr, movie_idx_to_id, feature_idx_to_id):
    n_movies = get_nrows_csr(features_by_movie_csr)
    n_features = len(feature_idx_to_id)

    # Degrees
    features_per_movie = get_nelements_by_row_csr(features_by_movie_csr)
    movies_per_feature = get_nelements_by_row_csr(movies_by_feature_csr)

    nnz = features_per_movie.sum()
    sparsity = nnz / (n_movies * n_features)

    print(f"Number of movies: {n_movies}")
    print(f"Number of features: {n_features}")
    print(f"Total non-zero entries: {nnz}")
    print(f"Sparsity: {sparsity:.4f}")
    print(f"features:{feature_idx_to_id}")

In [ ]:
compute_csr_statistics(
    features_by_movie_csr,
    movies_by_feature_csr,
    movie_idx_to_id,
    feature_idx_to_id
)

In [ ]:
def compute_csr_statistics(
    features_by_movie_csr,
    movies_by_feature_csr,
    users_by_movie_csr,
    movies_by_user_csr,
    movie_idx_to_id,
    feature_idx_to_id,
    user_idx_to_id,
    bins=[0, 100, 200, 500, 1000, np.inf]
):
    n_movies = get_nrows_csr(features_by_movie_csr)
    n_features = len(feature_idx_to_id)
    n_users = get_nrows_csr(movies_by_user_csr)

    # Get degree-frequency distributions
    x_fpm, y_fpm = get_degree_frequency_csr(features_by_movie_csr)  # features per movie
    x_mpf, y_mpf = get_degree_frequency_csr(movies_by_feature_csr)  # movies per feature
    x_upm, y_upm = get_degree_frequency_csr(users_by_movie_csr)  # users per movie
    x_mpu, y_mpu = get_degree_frequency_csr(movies_by_user_csr)  # movies per user
    nratings =  get_nelements_by_row_csr(users_by_movie_csr).sum()

    print(f"Number of movies: {n_movies}")
    print(f"Number of features: {n_features}")
    print(f"Number of users: {n_users}")

    # Compute binned counts using get_values_per_bin
    print("\nRatings per movie:")
    bin_labels, bin_totals = get_values_per_bin(x_upm, y_upm, bins)
    for lbl, val in zip(bin_labels, bin_totals):
        print(f"{lbl}: {val}({val / n_movies * 100:.2f}%)")

    print("\nRatings per user:")
    bin_labels, bin_totals = get_values_per_bin(x_mpu, y_mpu, bins)
    for lbl, val in zip(bin_labels, bin_totals):
        print(f"{lbl}: {val}({val / n_users *100:.2f}%)")

    print(f"\nTotal ratings:{nratings}")
    print(f"features:{feature_idx_to_id}")
    stats = {
        "features_per_movie": (x_fpm, y_fpm),
        "movies_per_feature": (x_mpf, y_mpf),
        "users_per_movie": (x_upm, y_upm),
        "movies_per_user": (x_mpu, y_mpu),
        "bins": bins
    }
    return stats


In [ ]:
_ = compute_csr_statistics(
    features_by_movie_csr,
    movies_by_feature_csr,
    ratings_by_movie_csr,
    ratings_by_user_csr,
    movie_idx_to_id,
    feature_idx_to_id,
    user_idx_to_id,
    bins=[0, 10**2, 10**3, 10**4, np.inf]
)

In [ ]:
import numpy as np

def get_distribution_stats(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)

    if np.sum(y) == 0:
        raise ValueError("Sum of frequencies y is zero.")

    # Weighted mean
    mean_val = np.average(x, weights=y)

    # Weighted variance
    variance_val = np.average((x - mean_val)**2, weights=y)

    # Weighted skewness
    skewness_val = np.average((x - mean_val)**3, weights=y) / (np.sqrt(variance_val)**3)

    # Weighted kurtosis (excess)
    kurtosis_val = np.average((x - mean_val)**4, weights=y) / (variance_val**2) - 3

    return {
        "mean": mean_val,
        "variance": variance_val,
        "skewness": skewness_val,
        "kurtosis": kurtosis_val
    }


In [ ]:
x_degrees, y_frequencies = get_degree_frequency_csr(ratings_by_user_csr)
stats = get_distribution_stats(x_degrees, y_frequencies)

print("Ratings per user statistics:")
print(f"Mean: {stats['mean']:.2f}")
print(f"Variance: {stats['variance']:.2f}")
print(f"Skewness: {stats['skewness']:.2f}")
print(f"Kurtosis: {stats['kurtosis']:.2f}")

x_degrees, y_frequencies = get_degree_frequency_csr(ratings_by_movie_csr)
stats = get_distribution_stats(x_degrees, y_frequencies)

print("\nRatings per movie statistics:")
print(f"Mean: {stats['mean']:.2f}")
print(f"Variance: {stats['variance']:.2f}")
print(f"Skewness: {stats['skewness']:.2f}")
print(f"Kurtosis: {stats['kurtosis']:.2f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import textwrap

def plot_bar_general(x_labels, y_values, sort=True, topn=None, bw_plot=True,
                     xlabel="X", ylabel="Y", title="Bar Plot", rotate_xticks=45,
                     horizontal=False, max_height_per_label=0.5, max_label_width=20,
                     figsize=None):
    y_values = np.array(y_values)
    indices = np.arange(len(x_labels))

    # Sorting
    if sort:
        indices = np.flip(np.argsort(y_values))
    if topn is not None:
        indices = indices[:topn]

    x_labels_sorted = [x_labels[i] for i in indices]
    y_values_sorted = y_values[indices]

    # Wrap long labels
    wrapped_labels = ["\n".join(textwrap.wrap(label, max_label_width)) for label in x_labels_sorted]

    # Determine figure size
    n_bars = len(wrapped_labels)
    if figsize is None:
        if horizontal:
            fig_height = max(5, n_bars * max_height_per_label)
            fig = plt.figure(figsize=(10, fig_height))
        else:
            fig_width = max(8, n_bars * max_height_per_label)
            fig = plt.figure(figsize=(fig_width, 5))
    else:
        fig = plt.figure(figsize=figsize)

    ax = fig.add_subplot(1, 1, 1)

    # Bar color
    bar_color = 'lightgray' if bw_plot else 'C0'
    edge_color = 'black'

    if horizontal:
        ax.barh(range(n_bars), y_values_sorted, color=bar_color, edgecolor=edge_color)
        ax.set_yticks(range(n_bars))
        ax.set_yticklabels(wrapped_labels)
        ax.set_xlabel(ylabel)
        ax.set_ylabel(xlabel)
        ax.invert_yaxis()  # Largest value on top
    else:
        ax.bar(range(n_bars), y_values_sorted, color=bar_color, edgecolor=edge_color)
        ax.set_xticks(range(n_bars))
        ax.set_xticklabels(wrapped_labels, rotation=rotate_xticks, ha='right')
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)

    ax.set_title(title)
    ax.grid(False)
    plt.tight_layout()

    # Print top items
    for label, value in zip(x_labels_sorted, y_values_sorted):
        print(fr"{label}: & {value} \\")

    return fig, ax

def plot_feature_distribution_bar(movies_by_feature_csr, feature_idx_to_id, sort=True,
                                  topn=None,
                                  horizontal=True,
                                  bw_plot=True, figsize=None):
    # Compute number of movies per feature
    nmovies_per_feature = get_nelements_by_row_csr(movies_by_feature_csr)

    # Optional: sort by popularity
    indices = np.arange(len(feature_idx_to_id))

    feature_names = [feature_idx_to_id[i] for i in indices]

    fig, ax = plot_bar_general(
        feature_names,
        nmovies_per_feature,
        sort=sort,
        topn=topn,
        horizontal=horizontal,
        bw_plot=bw_plot,
        xlabel="Feature",
        ylabel="Number of Movies",
        title="Number of Movies per Feature",
        figsize=figsize
    )
    return fig, ax

In [ ]:
plot_folder = Path(".") / "plots"
filename = "Movie-over-Feature-Distribution.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)

plot_feature_distribution_bar(movies_by_feature_csr, feature_idx_to_id, sort=True,
                              figsize=(6,10))

save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

In [ ]:
degrees = get_nelements_by_row_csr(ratings_by_movie_csr)
ids = get_id_from_idx(range(len(degrees)), movie_idx_to_id)
titles = get_movie_title_from_id(ids, movie_id_to_idx, movies_title)

plot_bar_general(
    titles,
    degrees,
    topn=20,
    sort=True,
    bw_plot=True,
    # horizontal=True,
    max_label_width=50,
    max_height_per_label=100,
    horizontal=True,
    figsize=(5,9),
    rotate_xticks=90,
    xlabel="Movie",
    ylabel="Number of Ratings",
    title="Top 20 Movies by Number of Ratings"
)

# Practical 4

Split CSR

In [ ]:
@show_execution_duration()
def parse_randomly_train_test_split_data_csr(
    ratings_path: str,
    split: float = 0.9,
    seed=42):
    """
    Parse ratings file and produce efficient array-based CSR-style structures.
    Numba-ready.
    """
    np.random.seed(seed)
    # Maps
    user_id_to_idx = {}
    movie_id_to_idx = []

    user_id_to_idx = {}
    user_idx_to_id = []
    movie_id_to_idx = {}
    movie_idx_to_id = []

    # Temporary buckets for building CSR
    ratings_by_user_train = []     # each element: list[(movie_idx, rating)]
    ratings_by_user_test = []
    ratings_by_movie_train = []
    ratings_by_movie_test = []
    total = int(subprocess.check_output(["wc", "-l", ratings_path]).split()[0])
    with open(ratings_path, "r", encoding="UTF-8") as f:

        next(f)

        for line in tqdm(f,total=total, unit=" lines"):
            userId, movieId, rating, _ = line.split(",")
            userId = int(userId)
            movieId = int(movieId)
            rating = float(rating)

            # Map user ID
            if userId not in user_id_to_idx:
                uidx = len(user_id_to_idx)
                user_id_to_idx[userId] = uidx
                user_idx_to_id.append(userId)
                ratings_by_user_train.append([])
                ratings_by_user_test.append([])

            uidx = user_id_to_idx[userId]

            # Map movie ID
            if movieId not in movie_id_to_idx:
                midx = len(movie_id_to_idx)
                movie_id_to_idx[movieId] = midx
                movie_idx_to_id.append(movieId)
                ratings_by_movie_train.append([])
                ratings_by_movie_test.append([])

            midx = movie_id_to_idx[movieId]

            # Random split
            if np.random.random() < split:
                ratings_by_user_train[uidx].append((midx, rating))
                ratings_by_movie_train[midx].append((uidx, rating))
            else:
                ratings_by_user_test[uidx].append((midx, rating))
                ratings_by_movie_test[midx].append((uidx, rating))

    # Convert to CSR arrays

    ratings_by_user_train_csr = build_csr(ratings_by_user_train)
    ratings_by_user_test_csr  = build_csr(ratings_by_user_test)
    ratings_by_movie_train_csr = build_csr(ratings_by_movie_train)
    ratings_by_movie_test_csr  = build_csr(ratings_by_movie_test)

    return (
        user_id_to_idx,
        user_idx_to_id,
        movie_id_to_idx,
        movie_idx_to_id,
        ratings_by_user_train_csr,
        ratings_by_movie_train_csr,
        ratings_by_user_test_csr,
        ratings_by_movie_test_csr,
    )

In [ ]:
@show_execution_duration()
def parse_with_feature_randomly_train_test_split_data_csr(
    ratings_path: str,
    data_movies_path:str,
    split: float = 0.9,
    seed=None):
    """
    Parse ratings file and produce efficient array-based CSR-style structures.
    Numba-ready.
    """
    np.random.seed(seed)
    # Maps
    user_id_to_idx = {}
    movie_id_to_idx = []

    user_id_to_idx = {}
    user_idx_to_id = []
    movie_id_to_idx = {}
    movie_idx_to_id = []

    # Temporary buckets for building CSR
    ratings_by_user_train = []
    ratings_by_user_test = []
    ratings_by_movie_train = []
    ratings_by_movie_test = []


    total = int(subprocess.check_output(["wc", "-l", ratings_path]).split()[0])
    with open(ratings_path, "r", encoding="UTF-8") as f:

        next(f)

        for line in tqdm(f,total=total, unit=" lines",  desc="Loading ratings"):
            userId, movieId, rating, _ = line.split(",")
            userId = int(userId)
            movieId = int(movieId)
            rating = float(rating)

            # Map user ID
            if userId not in user_id_to_idx:
                uidx = len(user_id_to_idx)
                user_id_to_idx[userId] = uidx
                user_idx_to_id.append(userId)
                ratings_by_user_train.append([])
                ratings_by_user_test.append([])

            uidx = user_id_to_idx[userId]

            # Map movie ID
            if movieId not in movie_id_to_idx:
                midx = len(movie_id_to_idx)
                movie_id_to_idx[movieId] = midx
                movie_idx_to_id.append(movieId)
                ratings_by_movie_train.append([])
                ratings_by_movie_test.append([])

            midx = movie_id_to_idx[movieId]

            # Random split
            if np.random.random() < split:
                ratings_by_user_train[uidx].append((midx, rating))
                ratings_by_movie_train[midx].append((uidx, rating))
            else:
                ratings_by_user_test[uidx].append((midx, rating))
                ratings_by_movie_test[midx].append((uidx, rating))

        feature_id_to_idx = {}
        feature_idx_to_id = []

        movies_title = [ [] for _ in range(len(movie_idx_to_id))]
        movies_by_feature = []
        features_by_movie = [ [] for _ in range(len(movie_idx_to_id))]

        total = int(subprocess.check_output(["wc", "-l", data_movies_path]).split()[0]) - 1
        with open(data_movies_path, "r", encoding="UTF-8") as f:
            next(f)

            for line in tqdm(f,total=total, unit=" lines", desc="Loading features"):
                movieId, title, features_str =  next(csv.reader([line]))
                movieId = int(movieId)
                featuresIds = features_str.split("|")
                if movieId in movie_id_to_idx:
                    midx = movie_id_to_idx[movieId]
                    movies_title[midx] = title
                    # Map Features ID
                    for featureId in featuresIds:
                        if featureId not in feature_id_to_idx:
                            fidx = len(feature_id_to_idx)
                            feature_id_to_idx[featureId] = fidx
                            feature_idx_to_id.append(featureId)
                            movies_by_feature.append([])
                        fidx = feature_id_to_idx[featureId]
                        movies_by_feature[fidx].append((midx, 1))
                        features_by_movie[midx].append((fidx, 1))

    # Convert to CSR arrays

    ratings_by_user_train_csr = build_csr(ratings_by_user_train)
    ratings_by_user_test_csr  = build_csr(ratings_by_user_test)
    ratings_by_movie_train_csr = build_csr(ratings_by_movie_train)
    ratings_by_movie_test_csr  = build_csr(ratings_by_movie_test)

    movies_by_feature_csr = build_csr(movies_by_feature)
    features_by_movie_csr = build_csr(features_by_movie)

    return (
        user_id_to_idx,
        user_idx_to_id,
        movie_id_to_idx,
        movie_idx_to_id,
        feature_idx_to_id,
        feature_id_to_idx,
        ratings_by_user_train_csr,
        ratings_by_movie_train_csr,
        ratings_by_user_test_csr,
        ratings_by_movie_test_csr,
        features_by_movie_csr,
        movies_by_feature_csr,
        movies_title,
    )

In [ ]:
# (
#     user_id_to_idx, user_idx_to_id, movie_id_to_idx, movie_idx_to_id,
#     ratings_by_user_train_csr, ratings_by_movie_train_csr,
#     ratings_by_user_test_csr, ratings_by_movie_test_csr
# )= parse_randomly_train_test_split_data_csr(ratings_path)
# assert get_nrows_csr(ratings_by_user_train_csr) == get_nrows_csr(ratings_by_user_test_csr)
# assert get_nrows_csr(ratings_by_movie_train_csr) == get_nrows_csr(ratings_by_movie_test_csr)
(
    user_id_to_idx, user_idx_to_id, movie_id_to_idx, movie_idx_to_id,
    feature_idx_to_id, feature_id_to_idx,
    ratings_by_user_train_csr, ratings_by_movie_train_csr,
    ratings_by_user_test_csr, ratings_by_movie_test_csr,
    features_by_movie_csr, movies_by_feature_csr, movies_title
)= parse_with_feature_randomly_train_test_split_data_csr(ratings_path, movies_path, split=0.9, seed=42)
assert get_nrows_csr(ratings_by_user_train_csr) == get_nrows_csr(ratings_by_user_test_csr)
assert get_nrows_csr(ratings_by_movie_train_csr) == get_nrows_csr(ratings_by_movie_test_csr) == get_nrows_csr(features_by_movie_csr)

Train

In [ ]:
# power law Users
type_vector = "Users"
datasplit_to = "train"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "
plot_folder = Path(".") / "plots"
filename = f"power-law-{datasplit_to}-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_user_train_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C0',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

# a, gamma = get_power_law_from_csr(ratings_by_user_train_csr)
# print(f"{type_vector}: p(d) proportional to  d^({gamma:.2f})")

In [ ]:
# power law Movies
type_vector = "Movies"
datasplit_to = "train"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "

plot_folder = Path(".") / "plots"
filename = f"power-law-{datasplit_to}-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_movie_train_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C1',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

# a, gamma = get_power_law_from_csr(ratings_by_movie_test_csr)
# print(f"{type_vector}: p(d) proportional to  d^({gamma:.2f})")

Test

In [ ]:
# power law Users
type_vector = "Users"
datasplit_to = "test"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "
plot_folder = Path(".") / "plots"
filename = f"power-law-{datasplit_to}-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_user_test_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C0',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

# a, gamma = get_power_law_from_csr(ratings_by_user_test_csr)
# print(f"{type_vector}: p(d) proportional to  d^({gamma:.2f})")

In [ ]:
# power law Movies
type_vector = "Movies"
datasplit_to = "test"
title = f"Ratings: {type_vector} Degree Distribution"
title = " "

plot_folder = Path(".") / "plots"
filename = f"power-law-{datasplit_to}-{type_vector}.pdf"
plot_filepath = plot_folder / filename
plot_filepath = str(plot_filepath)
fig, ax = plt.subplots(1, figsize=(6,6))

plot_ratings_csr_publication(ratings_by_movie_test_csr,
                            title=title,
                            log_plot=True,
                            bw_plot=False,
                            color='C1',
                            fig_ax=(fig, ax))
save_plot(
    plot_filepath,
    fig=None,
    formats="pdf",
    timestamps=False,
    use_tex=False,
    dpi=300,
    overwrite=True
)

# a, gamma = get_power_law_from_csr(ratings_by_movie_train_csr)
# print(f"{type_vector}: p(d) proportional to  d^({gamma:.2f})")

In [ ]:
def plot_als_curve_only(train_hist:np.ndarray,
                    test_hist:np.ndarray,
                    title="",
                    ylabel="",
                    suptitle=None,
                    fig_ax=None,
                    figsize=(6,4))-> None:
    epoch_range = range(1, len(train_hist) + 1)
    if fig_ax:
        fig, ax = fig_ax
    else:
        fig, ax = plt.subplots(figsize=figsize)
    ax.plot(epoch_range, train_hist,marker="", label="train")
    ax.plot(epoch_range, test_hist, marker="", ls='--',  label="test")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.set_ylabel(ylabel)
    ax.legend()

    fig.suptitle(suptitle)
    fig.tight_layout()
    return fig, ax

def plot_als_curves(train_loss_hist:np.ndarray,
                    test_loss_hist:np.ndarray,
                    train_rmse_hist:np.ndarray,
                    test_rmse_hist:np.ndarray,
                    K:int, precision:float, vector_penality:float, bias_penality:float,
                    bias_only=False,
                    fig_ax=None,
                    toplot="both",
                    with_suptitle=True)-> None:
    epoch_range = range(1, len(train_rmse_hist) + 1)
    plot_loss = False
    plot_rmse = False
    if toplot=="both":
        fig, axes = plt.subplots(1, 2, figsize=(12,4), sharex=True)
        ax_loss = axes[0]
        ax_rmse = axes[1]
        plot_loss = True
        plot_rmse = True
        fig_ax =  fig, axes
    else:
        if fig_ax:
            fig, ax = fig_ax
        else:
            fig, ax = plt.subplots(figsize=(6,4))
            fig_ax =  fig, ax
        if toplot=="loss":
            plot_loss = True
            ax_loss = ax

        elif toplot=="rmse":
            plot_rmse = True
            ax_rmse = ax
    if plot_loss:
        title = "Loss"
        ylabel = "loss"
        plot_als_curve_only(train_loss_hist,
                        test_loss_hist,
                        title=title,
                        ylabel=ylabel,
                        fig_ax=(fig, ax_loss),
                        )
    if plot_rmse:
        title = "RMSE"
        ylabel = "rmse"
        plot_als_curve_only(train_rmse_hist,
                        test_rmse_hist,
                        title=title,
                        ylabel=ylabel,
                        fig_ax=(fig, ax_rmse),
                        figsize=(6,4)
                        )

    if with_suptitle:
        if bias_only:
            fig.suptitle(fr"$K$={K}, $\lambda$={precision}, $\tau_{{bias}}$={bias_penality}")
        else:
            fig.suptitle(fr"$K$={K}, $\lambda$={precision}, $\tau$={vector_penality}, $\tau_{{bias}}$={bias_penality}")
        fig.tight_layout()
    return fig_ax

### Parallelized Numba list of tuple

### Parallelized CSR format data

bias only

In [ ]:
@njit
def get_row_from_csr(matrix_csr, row_idx):
    row_offset, col_indices, values = matrix_csr
    start = row_offset[row_idx]
    end = row_offset[row_idx + 1]
    return col_indices[start:end], values[start:end]

@njit
def get_nrows_csr(matrix_csr):
    row_offset, _, _ = matrix_csr
    return len(row_offset) - 1

@njit(fastmath=True)
def calc_metrics_numba_nb_csr_bias_only(
        user_biases:np.ndarray,
        item_biases:np.ndarray,
        ratings_by_user_csr:tuple[np.ndarray, np.ndarray, np.ndarray],
        precision:float,
        bias_penality:float
        ):
    M = get_nrows_csr(ratings_by_user_csr)

    se = 0.0
    se_biases = 0.0
    cnt = 0


    for m in range(M):
        items, ratings = get_row_from_csr(ratings_by_user_csr, m)
        nitems = len(items)
        if nitems == 0:
            continue

        pred = item_biases[items]  + user_biases[m] # shape (nitems,)
        err = ratings - pred
        err_biases = ratings - user_biases[m] - item_biases[items]

        se += np.sum(err ** 2)
        se_biases += np.sum(err_biases ** 2)
        cnt += nitems

    rmse = np.sqrt(se / cnt)
    loss = (
        precision / 2.0 * se_biases
        + bias_penality / 2.0 * (np.sum(user_biases ** 2) + np.sum(item_biases ** 2))
    )
    return rmse, loss

@njit(parallel=True, fastmath=True)
def als_numba_iteration_nb_csr_bias_only(user_biases, item_biases,
                           ratings_by_user_csr, ratings_by_item_csr,
                           precision, bias_penality, K):
    M = get_nrows_csr(ratings_by_user_csr)
    N = get_nrows_csr(ratings_by_item_csr)

    def update_user_bias_only(m: int):
        items, ratings = get_row_from_csr(ratings_by_user_csr, m)
        nitems = len(items)
        if nitems == 0:
            return user_biases[m]
        item_bias = item_biases[items]          # (nitems,)
        pred =  item_bias
        resid = ratings - pred

        s = precision * np.sum(resid)
        user_bias = s / (precision * nitems + bias_penality)
        return user_bias

    def update_item_bias_only(n: int):
        users, ratings = get_row_from_csr(ratings_by_item_csr, n)
        nusers = len(users)
        if nusers == 0:
            return item_biases[n]

        user_bias = user_biases[users]           # (nusers,)
        pred = user_bias
        resid = ratings - pred

        s = precision * np.sum(resid)
        item_bias = s / (precision * nusers + bias_penality)
        return item_bias


    for m in prange(M):
    #     user_biases[m] = update_user(m)
        user_biases[m] = update_user_bias_only(m)

    for n in prange(N):
        # item_biases[n] = update_item(n)
        item_biases[n]= update_item_bias_only(n)


@show_execution_duration(return_time=True)
def als_update_biases_nb_csr_bias_only(
    ratings_by_user_train_csr,
    ratings_by_movie_train_csr,
    ratings_by_user_test_csr,
    ratings_by_movie_test_csr,
    precision:float,
    bias_penality:float,
    K,
    max_iter=10
):
    M = get_nrows_csr(ratings_by_user_train_csr)
    N = get_nrows_csr(ratings_by_movie_train_csr)

    # Initialize embeddings and biases
    mean = np.zeros(K)
    cov = (1.0 / np.sqrt(K)) * np.identity(K)
    user_biases = np.zeros(M)
    item_biases = np.zeros(N)

    train_rmse_hist = []
    train_loss_hist = []
    test_rmse_hist = []
    test_loss_hist = []

    for it in tqdm(range(max_iter), unit="epoch"):
        als_numba_iteration_nb_csr_bias_only(
            user_biases, item_biases,
            ratings_by_user_train_csr, ratings_by_movie_train_csr,
            precision, bias_penality, K)

        train_rmse, train_loss = calc_metrics_numba_nb_csr_bias_only(
                                                       user_biases, item_biases,
                                                       ratings_by_user_train_csr,
                                                       precision, bias_penality)
        test_rmse, test_loss = calc_metrics_numba_nb_csr_bias_only(
                                                     user_biases, item_biases,
                                                     ratings_by_user_test_csr,
                                                     precision, bias_penality)

        train_rmse_hist.append(train_rmse)
        train_loss_hist.append(train_loss)
        test_rmse_hist.append(test_rmse)
        test_loss_hist.append(test_loss)

        if (it + 1) % 10 == 0 or it == 0:
            print("")
            print(f"epoch {it+1}/{max_iter} | train_loss={train_loss:.4f} "
                  f"test_loss={test_loss:.4f} | train_rmse={train_rmse:.4f} "
                  f"test_rmse={test_rmse:.4f}")

    return (user_biases, item_biases,
            train_loss_hist, train_rmse_hist, test_loss_hist, test_rmse_hist
            )

CSR Training

In [ ]:

# Training using numba csr v1.2 vectorized
@njit
def get_row_from_csr(matrix_csr, row_idx):
    row_offset, col_indices, values = matrix_csr
    start = row_offset[row_idx]
    end = row_offset[row_idx + 1]
    return col_indices[start:end], values[start:end]

@njit
def get_nrows_csr(matrix_csr):
    row_offset, _, _ = matrix_csr
    return len(row_offset) - 1

@njit(fastmath=True)
def calc_metrics_numba_nb_csr(
        user_vectors:np.ndarray[np.ndarray],
        item_vectors:np.ndarray[np.ndarray],
        user_biases:np.ndarray,
        item_biases:np.ndarray,
        ratings_by_user_csr:tuple[np.ndarray, np.ndarray, np.ndarray],
        precision:float,
        vector_penality:float,
        bias_penality:float):
    M = get_nrows_csr(ratings_by_user_csr)

    se = 0.0
    se_biases = 0.0
    cnt = 0


    for m in range(M):
        items, ratings = get_row_from_csr(ratings_by_user_csr, m)
        nitems = len(items)
        if nitems == 0:
            continue

        V = item_vectors[items]             # (nitems, K)
        u = user_vectors[m]                 # (K,)
        pred = V @ u + item_biases[items]  + user_biases[m] # shape (nitems,)
        err = ratings - pred
        err_biases = ratings - V @ u - user_biases[m] - item_biases[items]

        se += np.sum(err ** 2)
        se_biases += np.sum(err_biases ** 2)
        cnt += nitems

    rmse = np.sqrt(se / cnt)
    loss = (
        precision / 2.0 * se_biases
        + bias_penality / 2.0 * (np.sum(user_biases ** 2) + np.sum(item_biases ** 2))
        + vector_penality / 2.0 * (np.sum(user_vectors ** 2) + np.sum(item_vectors ** 2))
    )
    return rmse, loss

@njit(parallel=True, fastmath=True)
def als_numba_iteration_nb_csr(
        user_vectors,
        item_vectors,
        user_biases,
        item_biases,
        ratings_by_user_csr,
        ratings_by_item_csr,
        precision,
        vector_penality,
        bias_penality,
        K):
    M = get_nrows_csr(ratings_by_user_csr)
    N = get_nrows_csr(ratings_by_item_csr)

    def update_user(m: int):
        items, ratings = get_row_from_csr(ratings_by_user_csr, m)
        nitems = len(items)
        if nitems == 0:
            return user_biases[m], user_vectors[m]

        V = item_vectors[items]                 # (nitems, K)
        u = user_vectors[m]                     # (K,)
        item_bias = item_biases[items]          # (nitems,)
        # user_bias = user_biases[m]
        pred = V @ u.T + item_bias
        resid = ratings - pred

        s = precision * np.sum(resid)
        user_bias = s / (precision * nitems + bias_penality)

        resid2 = ratings - user_bias - item_bias

        linsum_v = precision * (V.T @ resid2.T)
        transition_matrix = vector_penality * np.identity(K) + precision * (V.T @ V)

        user_vector = np.linalg.solve(transition_matrix, linsum_v)
        return user_bias, user_vector

    def update_item(n: int):
        users, ratings = get_row_from_csr(ratings_by_item_csr, n)
        nusers = len(users)
        if nusers == 0:
            return item_biases[n], item_vectors[n]

        U = user_vectors[users]                  # (nusers, K)
        v = item_vectors[n]                      # (K,)
        user_bias = user_biases[users]           # (nusers,)
        # item_bias = item_biases[n]
        pred = U @ v.T + user_bias
        resid = ratings - pred

        s = precision * np.sum(resid)
        item_bias = s / (precision * nusers + bias_penality)

        resid2 = ratings - user_bias- item_bias

        linsum_u = precision * (U.T @ resid2.T)
        transition_matrix = vector_penality * np.identity(K) + precision * (U.T @ U)

        item_vector = np.linalg.solve(transition_matrix, linsum_u)
        return item_bias, item_vector


    for m in prange(M):
    #     user_biases[m] = update_user(m)
        user_biases[m], user_vectors[m, :] = update_user(m)

    for n in prange(N):
        # item_biases[n] = update_item(n)
        item_biases[n], item_vectors[n, :] = update_item(n)


@show_execution_duration(return_time=True)
def als_update_embeddings_and_biases_nb_csr(
    ratings_by_user_train_csr,
    ratings_by_movie_train_csr,
    ratings_by_user_test_csr,
    ratings_by_movie_test_csr,
    precision,
    bias_penality,
    vector_penality,
    K,
    max_iter=10,
    seed=None
):
    M = get_nrows_csr(ratings_by_user_train_csr)
    N = get_nrows_csr(ratings_by_movie_train_csr)

    # Initialize embeddings and biases
    np.random.seed(seed)
    mean = np.zeros(K)
    cov = (1.0 / np.sqrt(K)) * np.identity(K)
    user_vectors = np.random.multivariate_normal(mean, cov, M)
    item_vectors = np.random.multivariate_normal(mean, cov, N)
    user_biases = np.zeros(M)
    item_biases = np.zeros(N)

    train_rmse_hist = []
    train_loss_hist = []
    test_rmse_hist = []
    test_loss_hist = []

    for it in tqdm(range(max_iter), unit="epoch"):
        als_numba_iteration_nb_csr(user_vectors, item_vectors, user_biases, item_biases,
                               ratings_by_user_train_csr, ratings_by_movie_train_csr,
                               precision, vector_penality, bias_penality, K)

        train_rmse, train_loss = calc_metrics_numba_nb_csr(user_vectors, item_vectors,
                                                       user_biases, item_biases,
                                                       ratings_by_user_train_csr,
                                                       precision, vector_penality, bias_penality)
        test_rmse, test_loss = calc_metrics_numba_nb_csr(user_vectors, item_vectors,
                                                     user_biases, item_biases,
                                                     ratings_by_user_test_csr,
                                                     precision, vector_penality, bias_penality)

        train_rmse_hist.append(train_rmse)
        train_loss_hist.append(train_loss)
        test_rmse_hist.append(test_rmse)
        test_loss_hist.append(test_loss)

        if (it + 1) % 10 == 0 or it == 0:
            print("")
            print(f"epoch {it+1}/{max_iter} | train_loss={train_loss:.4f} "
                  f"test_loss={test_loss:.4f} | train_rmse={train_rmse:.4f} "
                  f"test_rmse={test_rmse:.4f}")

    return (user_vectors, user_biases, item_vectors, item_biases,
            train_loss_hist, train_rmse_hist, test_loss_hist, test_rmse_hist
            )

In [ ]:
@show_execution_duration("Big Loop")
def train(K_range, vector_penality_range, bias_only=False, save=False):
    with_embeddings_path = model_folder / data_size_str/ "with_embeddings"
    bias_only_path = model_folder / data_size_str / "bias_only"
    for K in K_range:
        user_vectors = []
        item_vectors = []
        def fun_save():
            save_relative_path = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-lambda{precision}-tau{vector_penality}.joblib"
            save_relative_path_latest = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-latest.joblib"
            if bias_only:
                save_relative_path = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-lambda{precision}-tau{vector_penality}-bias_only.joblib"
                save_relative_path_latest = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-latest-bias_only.joblib"
                save_path =  bias_only_path / save_relative_path
                save_path_latest = bias_only_path / save_relative_path_latest
                suffix = f"iter{max_iter}-bias_only"
            else:
                save_relative_path = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-lambda{precision}-tau{vector_penality}.joblib"
                save_relative_path_latest = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-latest.joblib"
                save_path =  with_embeddings_path / save_relative_path
                save_path_latest = with_embeddings_path / save_relative_path_latest
                suffix = f"iter{max_iter}-with_embeddings"

            log_file = PROJECT_PATH / f"models/als_runs_{suffix}.yaml"
            log_file = log_file.as_posix()  # single YAML log for all joblib models
            data_model = {
                "user_vectors":user_vectors,
                "item_vectors":item_vectors,
                "user_biases":user_biases,
                "item_biases":item_biases,
                "precision":precision,
                "bias_penality":bias_penality,
                "vector_penality":vector_penality,
                "K":K,
                "bias_only":bias_only,
                "train_loss_hist":train_loss_hist,
                "train_rmse_hist":train_rmse_hist,
                "test_loss_hist":test_loss_hist,
                "test_rmse_hist":test_rmse_hist,
                "max_iter":max_iter,
                "training_starting_time":training_starting_time,
                "training_duration": training_duration,
                "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            }
            args =  {
                "precision":precision,
                "bias_penality":bias_penality,
                "vector_penality":vector_penality,
                "K":K,
                "bias_only":bias_only,
                # "train_loss_hist":train_loss_hist,
                "train_rmse_hist":train_rmse_hist[-2:],
                # "test_loss_hist":test_loss_hist,
                "test_rmse_hist":test_rmse_hist[-2:],
                "max_iter":max_iter,
                "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
                "model_path":save_relative_path

            }

            # Dump with timestamp
            saved_path = dump_objects(
                data_model,
                path= save_path,
                timestamp=True
            )

            run_id, log_file = log_als_run_to_yaml(log_file, args)

            print("Logged run:", run_id)
            print("YAML log file:", log_file)

            saved_path_latest = dump_objects(
                data_model,
                path= save_path_latest,
                timestamp=False
            )

        if bias_only:
            vector_penality = vector_penality_range[0]
            print(f"K:{K}")
            (
                (user_biases, item_biases, train_loss_hist,
                train_rmse_hist, test_loss_hist, test_rmse_hist), training_starting_time, training_duration
            ) = als_update_biases_nb_csr_bias_only(
                ratings_by_user_train_csr,
                ratings_by_movie_train_csr,
                ratings_by_user_test_csr,
                ratings_by_movie_test_csr,
                precision,
                bias_penality,
                K,
                max_iter=max_iter
            )
            if save:
                fun_save()
        else:
            for vector_penality in vector_penality_range:
                print(f"K:{K}, lambda:{1}, tau:{vector_penality}")
                (
                    (user_vectors, user_biases, item_vectors, item_biases,
                    train_loss_hist, train_rmse_hist, test_loss_hist, test_rmse_hist
                    ), training_starting_time, training_duration
                ) = als_update_embeddings_and_biases_nb_csr(ratings_by_user_train_csr,
                    ratings_by_movie_train_csr,
                    ratings_by_user_test_csr,
                    ratings_by_movie_test_csr,
                    precision,
                    bias_penality,
                    vector_penality,
                    K,
                    max_iter=max_iter,
                    seed=None
                )

                plot_als_curves(train_loss_hist,
                        test_loss_hist,
                        train_rmse_hist,
                        test_rmse_hist,
                        K,
                        precision,
                        vector_penality,
                        bias_penality,
                        bias_only=bias_only)
                if save:
                    fun_save()
    # return data_model

In [ ]:
precision = 1
bias_penality = 1e-3
# vector_penality_range = [0.1, 1, 10]
vector_penality_range = [1]
# K_range=[32, 16, 8, 4, 2]
K_range = [16, 8]
max_iter = 50
train(K_range, vector_penality_range, bias_only=True, save=True)

### Load

In [ ]:
def load_model(model_path:str)-> dict:
    results = load_objects(model_path)
    user_vectors = results["user_vectors"]
    item_vectors =  results["item_vectors"]
    user_biases = results["user_biases"]
    item_biases = results["item_biases"]
    train_loss_hist = results["train_loss_hist"]
    test_loss_hist = results["test_loss_hist"]
    train_loss_hist = results["train_loss_hist"]
    train_rmse_hist = results["train_rmse_hist"]
    test_rmse_hist = results["test_rmse_hist"]
    precision = results["precision"]
    bias_penality = results["bias_penality"]
    vector_penality = results["vector_penality"]
    K = results["K"]
    if "feature_vectors" in results:
        feature_vectors = results["feature_vectors"]
    else:
        feature_vectors = []
    data_model = {
        "user_vectors":user_vectors,
        "item_vectors":item_vectors,
        "user_biases":user_biases,
        "item_biases":item_biases,
        "feature_vectors":feature_vectors,
        "precision":precision,
        "bias_penality":bias_penality,
        "vector_penality":vector_penality,
        "K":K,
        "train_loss_hist":train_loss_hist,
        "train_rmse_hist":train_rmse_hist,
        "test_loss_hist":test_loss_hist,
        "test_rmse_hist":test_rmse_hist,
    }
    return data_model

In [ ]:
from pathlib import Path
from typing import Iterable

In [ ]:
def load_model_given_K(model_folder:str, K:int=16, pattern="", exclude_pattern=None):

    model_path_pattern =f".*K{K:02d}.*"
    if exclude_pattern:
        combined_pattern = rf"^(?=.*{model_path_pattern})(?=.*{pattern})(?!.*{exclude_pattern}).*$"
    else:
        combined_pattern = rf"^(?=.*{model_path_pattern})(?=.*{pattern}).*$"
    models_path = get_paths(model_folder, combined_pattern)
    print(models_path)
    models = []
    for path in models_path:
        models.append(load_model(path))
    return models

In [ ]:
K = 16
features=True
if features:
    models_path = model_folder / "features" / "ml-32m" / "with_embeddings"
else:
    models_path = model_folder / "ml-32m" / "with_embeddings"

exclude_pattern = "latest.joblib$"
models_K =  load_model_given_K(models_path, K, exclude_pattern=exclude_pattern)
fitted_model = models_K[-1]

In [ ]:
def plot_als_curves_multiple_models(fitted_models:list[dict],
                    fig_ax=None,
                    toplot="both",
                    with_suptitle=True):
    for model in fitted_models:
        if len(model["user_vectors"]):
            bias_only = False
        else:
            bias_only = True
        train_loss_hist = model["train_loss_hist"]
        test_loss_hist = model["test_loss_hist"]
        train_loss_hist = model["train_loss_hist"]
        train_rmse_hist = model["train_rmse_hist"]
        test_rmse_hist = model["test_rmse_hist"]
        precision = model["precision"]
        vector_penality = model["vector_penality"]
        bias_penality = model["bias_penality"]
        K = model["K"]
        fig_ax = plot_als_curves(
            train_loss_hist,
            test_loss_hist,
            train_rmse_hist,
            test_rmse_hist,
            K,
            precision,
            vector_penality,
            bias_penality,
            bias_only=bias_only,
            fig_ax=fig_ax,
            toplot=toplot,
            with_suptitle=with_suptitle)

In [ ]:
def save_model_metrics(fitted_model, toplot, plot_folder, save=False, overwrite=False):
    precision = fitted_model["precision"]
    vector_penality = fitted_model["vector_penality"]
    bias_penality = fitted_model["bias_penality"]
    K = fitted_model["K"]
    max_iter = len(fitted_model["train_loss_hist"])

    if len(fitted_model["user_vectors"]):
        modeltype = "with-embeddings"
    else:
        modeltype = "bias-only"
    if not plot_folder:
        plot_folder = Path(".") / "plots" / "models"
    filename = f"{toplot}-model-{modeltype}-K{K}-iter{max_iter}-{precision}-{vector_penality}-{bias_penality}.pdf"
    plot_filepath = plot_folder / filename
    plot_filepath = str(plot_filepath)

    fig, ax = plt.subplots(1, figsize=(6,4))

    plot_als_curves_multiple_models(
        [fitted_model],
        fig_ax=(fig, ax),
        toplot=toplot,
        with_suptitle=True)
    ax.set_title(None)

    if save:
        save_plot(
            plot_filepath,
            fig=None,
            formats="pdf",
            timestamps=False,
            use_tex=False,
            dpi=300,
            overwrite=overwrite
        )
        print(f"Plot save to {plot_filepath}")
    return plot_filepath


In [ ]:
def get_models_metrics(models_path:list[str]):
    train_last_loss_list = []
    test_last_loss_list = []
    train_last_rmse_list = []
    test_last_rmse_list = []
    precision_list = []
    vector_penality_list = []
    bias_penality_list = []
    K_list = []
    iteration_list = []
    bias_only_list = []

    for path in models_path:
        model = load_model(path)
        if len(model["user_vectors"]):
            bias_only_list.append(False)
        else:
            bias_only_list.append(True)
        train_last_loss_list.append(model["train_loss_hist"][-1])
        test_last_loss_list.append(model["test_loss_hist"][-1])
        train_last_rmse_list.append(model["train_rmse_hist"][-1])
        test_last_rmse_list.append(model["test_rmse_hist"][-1])
        precision_list.append(model["precision"])
        vector_penality_list.append(model["vector_penality"])
        bias_penality_list.append(model["bias_penality"])
        K_list.append(model["K"])
        iteration_list.append(len(model["test_rmse_hist"]))
    return (
    train_last_loss_list,
    test_last_loss_list,
    train_last_rmse_list,
    test_last_rmse_list,
    precision_list,
    vector_penality_list,
    bias_penality_list,
    K_list,
    iteration_list,
    bias_only_list
    )


In [ ]:
!pip install adjustText


In [ ]:
import matplotlib.pyplot as plt
from adjustText import adjust_text

def plot_metric_scatter(x, y, z, xlabel="K", ylabel=r"$\tau$", title="RMSE", fig_ax=None):
    if fig_ax:
        fig, ax = fig_ax
    else:
        fig, ax = plt.subplots(figsize=(8, 6))
        fig_ax = fig, ax

    # Scatter plot colored by z
    scatter = ax.scatter(x, y, c=z, cmap='viridis', s=80)

    # Create text labels
    texts = []
    for xi, yi, zi in zip(x, y, z):
        texts.append(ax.text(xi, yi, f"{zi:.3f}", fontsize=11))

    # Automatically adjust text to avoid overlap with arrows
    adjust_text(
        texts,
        ax=ax,
        arrowprops=dict(arrowstyle="->", color='gray', lw=0.7),
        force_points=0.5,
        force_text=0.5,
        expand_text=(1.2, 1.2)
    )

    # Set ticks only at actual x and y values
    ax.set_xticks(sorted(set(x)))
    ax.set_yticks(sorted(set(y)))
    # Labels and title
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    # Colorbar
    fig.colorbar(scatter, ax=ax, label="RMSE")

    # Grid and layout
    ax.grid(True)
    fig.tight_layout()
    return fig_ax

In [ ]:
features=True
if features:
    embeddings_models_folder_path = model_folder / "features" / "ml-32m" / "with_embeddings"
else:
    embeddings_models_folder_path = model_folder / "ml-32m" / "with_embeddings"

paths = list(embeddings_models_folder_path.iterdir())
(
    train_last_loss_list,
    test_last_loss_list,
    train_last_rmse_list,
    test_last_rmse_list,
    precision_list,
    vector_penality_list,
    bias_penality_list,
    K_list,
    iteration_list,
    bias_only_list
) = get_models_metrics(paths)


In [ ]:
def save_metrics_grid(
    K_list,
    vector_penality_list,
    test_last_rmse_list,
    plot_filepath,
    title="",
    save=True,
    overwrite=False,
):
    # Ensure the parent folder exists
    plot_filepath = Path(plot_filepath)
    plot_filepath.parent.mkdir(parents=True, exist_ok=True)

    # Create the plot (your function should return fig, ax)
    fig, ax = plot_metric_scatter(
        K_list,
        vector_penality_list,
        test_last_rmse_list,
        title=title,
        fig_ax=None
    )

    # Save if requested
    if save:
        save_plot(
            plot_filepath,
            fig=fig,
            formats="pdf",
            timestamps=False,
            use_tex=False,
            dpi=300,
            overwrite=overwrite
        )
        print(f"Plot saved to {plot_filepath}")

    return plot_filepath

In [ ]:
features = False
if features:
   feature_is = "features"
else:
    feature_is = ""
plot_folder = Path(".") / "plots" / "models"
modeltype = "with-embedding"
datasplit_to = "test"
toplot = "rmse"
plot_filepath = plot_folder / f"{feature_is}-{datasplit_to}-{toplot}-grid-ml-32m-{modeltype}"
title = f"Test {toplot} After {np.min(iteration_list)} Iterations"

save_metrics_grid(
    K_list,
    vector_penality_list,
    test_last_rmse_list,
    plot_filepath,
    title=title,
    save=True,
    overwrite=True
)

plot_folder = Path(".") / "plots" / "models"
modeltype = "with-embedding"
datasplit_to = "train"
toplot = "rmse"
plot_filepath = plot_folder / f"{feature_is}{datasplit_to}-{toplot}-grid-ml-32m-{modeltype}"
title = f"Test {toplot} After {np.min(iteration_list)} Iterations"

save_metrics_grid(
    K_list,
    vector_penality_list,
    train_last_rmse_list,
    plot_filepath,
    title=title,
    save=True,
    overwrite=True
)

Bias only

In [ ]:
path = "/content/drive/MyDrive/Labs/ml_scale/recommender/models/ml-32m/with_embeddings/ml-32m-als-K16-iter100-lambda1-tau10.joblib"
fitted_model = load_model(path)
toplot = "loss"
plot_folder = Path(".") / "plots" / "models/features"
save_model_metrics(fitted_model, toplot, plot_folder, save=True, overwrite=True)


plot_folder = Path(".") / "plots" / "models/features"
toplot = "rmse"
save_model_metrics(fitted_model, toplot, plot_folder, save=True, overwrite=True)

In [ ]:
fitted_model = models_K[-1]
toplot = "loss"
plot_folder = Path(".") / "plots" / "models"
save_model_metrics(fitted_model, toplot, plot_folder, save=True, overwrite=True)

In [ ]:
fitted_model = models_K[-1]
plot_folder = Path(".") / "plots" / "models"
toplot = "rmse"
save_model_metrics(fitted_model, toplot, plot_folder, save=True, overwrite=True)

Load data for embeddings plotting

In [ ]:
K_str = "16"
tau = 10
model_path_K02 = "/content/drive/MyDrive/Labs/ml_scale/recommender/models/ml-32m/with_embeddings/ml-32m-als-KK16-iter50-lambda1-tau10.joblib"
model_K02 = load_model(model_path_K02)
fitted_model = model_K02

In [ ]:
def plot_vectors_umap(vectors, labels=None, n_neighbors=15, min_dist=0.1):
    reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist)
    points = reducer.fit_transform(vectors)

    plt.figure(figsize=(8, 6))
    plt.scatter(points[:, 0], points[:, 1])

    if labels is not None:
        for (x, y), label in zip(points, labels):
            plt.text(x, y, str(label), fontsize=9)

    plt.title("UMAP projection")
    plt.grid(True)
    plt.show()

import matplotlib.pyplot as plt

def plot_2dvectors(vectors, labels=None, title="2D Vectors"):
    """
    Plot 2D vectors with optional labels.
    """
    import numpy as np
    vectors = np.array(vectors)
    if vectors.shape[1] != 2:
        raise ValueError("Input vectors must be 2-dimensional (n_samples x 2).")

    plt.figure(figsize=(8, 6))
    plt.scatter(vectors[:, 0], vectors[:, 1], s=3)

    if labels is not None:
        for (x, y), label in zip(vectors, labels):
            plt.text(x, y, str(label), fontsize=9)

    plt.title(title)
    plt.grid(True)
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.show()


In [ ]:
import re
from pathlib import Path

# Regex for timestamps:
# Matches _YYYY-MM-DD_HH-MM, _YYYY-MM-DD_HH-MM-SS, _YYYY-MM-DD_HH-MM-SS-ms
TIMESTAMP_REGEX = re.compile(
    r'_20\d{2}-\d{2}-\d{2}_\d{2}-\d{2}(-\d{2})?(-\d{1,3})?$'
)


def strip_timestamp_suffix(path: Path) -> Path:
    """Remove timestamp suffix from a filename, preserving extension."""
    stem = path.stem
    new_stem = TIMESTAMP_REGEX.sub('', stem)
    return path.with_name(new_stem + path.suffix)


def ensure_unique_path(path: Path) -> Path:
    """Avoid overwriting files by appending _1, _2, ... if necessary."""
    if not path.exists():
        return path

    parent = path.parent
    stem = path.stem
    suffix = path.suffix
    counter = 1

    while True:
        candidate = parent / f"{stem}_{counter}{suffix}"
        if not candidate.exists():
            return candidate
        counter += 1


def clean_filenames_recursive(folder: str):
    """
    Recursively clean all filenames in a folder by removing timestamp suffixes.

    Parameters:
        folder (str): Path to the root folder
    """
    folder = Path(folder)

    for file in folder.rglob("*"):
        if file.is_file():
            # Strip timestamp suffix
            new_path = strip_timestamp_suffix(file)

            # Only rename if changed
            if new_path != file:
                # Avoid collisions
                new_path = ensure_unique_path(new_path)

                file.rename(new_path)
                print(f"Renamed: {file.name} → {new_path.name}")

    print("Done! All filenames cleaned.")


### Predict

In [ ]:
def get_user_embedding(
        ratings_by_user,
        item_vectors:np.ndarray[np.ndarray],
        item_biases:np.ndarray,
        precision: float,
        bias_penality: float,
        vector_penality: float,
        max_iter: int = 50,
)-> tuple[np.ndarray, np.ndarray]:
    M = len(ratings_by_user)
    K = item_vectors.shape[1]
    mean = np.zeros(K)
    cov = (1.0 / np.sqrt(K)) * np.identity(K)
    user_vectors = np.random.multivariate_normal(mean, cov, M)
    user_biases = np.zeros(M)
    def update_user(m:int=0):
        items, ratings = zip(*ratings_by_user[m])
        items = list(items)
        ratings = list(ratings)
        nitems = len(items)
        if nitems == 0:
            return user_biases[m], user_vectors[m]

        V = item_vectors[items]                 # (nitems, K)
        u = user_vectors[m]                     # (K,)
        item_bias = item_biases[items]          # (nitems,)
        user_bias = user_biases[m]
        pred = V @ u + item_bias
        resid = ratings - pred

        s = precision * np.sum(resid)
        user_bias = s / (precision * nitems + bias_penality)

        resid2 = ratings - user_bias - item_bias

        linsum_v = precision * (resid2 @ V)
        transition_matrix = vector_penality * np.identity(K) + precision * (V.T @ V)

        user_vector = np.linalg.solve(transition_matrix, linsum_v)
        return user_bias, user_vector
    for it in range(max_iter):
        for m in range(M):
            user_biases[m], user_vectors[m, :] = update_user(m)

    return user_biases, user_vectors


def get_score_from_users_embedding(user_vectors:np.ndarray, item_vectors:np.ndarray,
                                user_biases:np.ndarray, item_biases:np.ndarray, bias_factor=0.05)->tuple[np.array, np.ndarray]:
    M = len(user_vectors)
    N = len(item_vectors)
    tems_score_by_user = np.empty((M, N))
    for m in range(M):
        tems_score_by_user[m, :] = item_vectors @ user_vectors[m].T + bias_factor * item_biases
    return tems_score_by_user

def argsort_array(list, descending=False):
    sort_idx = np.argsort(list)
    if descending:
        sort_idx = np.flip(sort_idx)
    return sort_idx

def get_top_recommended_item(
        ratings_by_user,
        movie_idx_to_id,
        item_vectors:np.ndarray[np.ndarray],
        item_biases:np.ndarray,
        precision: float,
        bias_penality: float,
        vector_penality: float,
        max_iter: int = 50,
        ntop=-1,
        bias_factor=0.05)->tuple[np.ndarray, np.ndarray]:
        M = len(ratings_by_user)
        N = len(item_vectors)
        user_biases, user_vectors = get_user_embedding(
            ratings_by_user,
            item_vectors,
            item_biases,
            precision,
            bias_penality,
            vector_penality,
            max_iter,
            )
        if ntop ==-1:
            ntop = N
        ntop_score_by_user = np.empty((M, ntop), dtype=float)
        ntop_item_ids_by_user = np.empty((M, ntop), dtype=int)

        items_score_by_user = get_score_from_users_embedding(
                user_vectors, item_vectors,
                user_biases, item_biases,
                bias_factor=bias_factor
                )

        for m in range(len(items_score_by_user)):
            scores = items_score_by_user[m]
            ntop_indices =  argsort_array(scores, descending=True)[:ntop]
            ntop_item_ids_by_user[m,:] = get_id_from_idx(ntop_indices, movie_idx_to_id)
            ntop_score_by_user[m,:] = scores[ntop_indices]

        return ntop_item_ids_by_user, ntop_score_by_user

In [ ]:
title = "Rambo"
ids, titles = find_movie_id_by_title(title, movies_title, movie_idx_to_id, movie_id_to_idx)
results = get_movie_info_from_id(
    ids,
    movie_id_to_idx,
    feature_idx_to_id,
    movies_title,
    features_by_movie_csr
    )
pd.DataFrame(results)

In [ ]:
# Toy Story
# dummy_ids = [120474]

# John Wick
dummy_ids = [115149]

# Spider-Man (2002)
dummy_ids = [5349]

# dummy_ids = [2402]

rated_movie_info = get_movie_info_from_id(
    dummy_ids, movie_id_to_idx, feature_idx_to_id,
    movies_title, features_by_movie_csr)
pd.DataFrame(rated_movie_info)

In [ ]:
dummy_idx = get_idx_from_id(dummy_ids, movie_id_to_idx)
dummy_ratings_by_user = [[(idx, 5) for idx in dummy_idx],]

bias_factor = 0.1
max_iter = 50
recommended_movies_ids, recommended_movie_score  = get_top_recommended_item(
        dummy_ratings_by_user,
        movie_idx_to_id,
        fitted_model['item_vectors'],
        fitted_model['item_biases'],
        fitted_model['precision'],
        fitted_model['bias_penality'],
        fitted_model['vector_penality'],
        max_iter=max_iter,
        ntop=20,
        bias_factor=bias_factor,
    )

recommended_info = []
for user_recommended_movie_id in recommended_movies_ids:
    recommended_movie_info = get_movie_info_from_id(
    user_recommended_movie_id, movie_id_to_idx,
    feature_idx_to_id,
    movies_title,
    features_by_movie_csr)

df = pd.DataFrame(recommended_movie_info).head(5)
df

In [ ]:
df.to_latex("a.txt")

### Filter movies Less than 100 ratings

In [ ]:
def get_idx_filter_by_number_of_rating(matrix_csr:tuple[np.ndarray, np.ndarray, np.ndarray],
                                       min_nratings=100) -> np.ndarray:
    degree = get_nelements_by_row_csr(matrix_csr)
    mask = degree >= min_nratings
    filtered_idx = np.array(range(len(degree)))[mask]
    return filtered_idx

In [ ]:
@njit
def csr_filter_values(matrix_csr:tuple[np.ndarray, np.ndarray, np.ndarray],
                      fun_mask:Callable[[list], list[bool]])->tuple[np.ndarray, np.ndarray, np.ndarray]:
    row_offset, col_indices, values = matrix_csr
    mask = fun_mask(values)
    new_row_offset= np.zeros_like(row_offset)
    new_col_idx = col_indices[mask]
    new_values = values[mask]

    # rebuild row_offset
    nnz = 0
    pos = 0
    for r in range(len(row_offset) - 1):
        start = row_offset[r]
        end = row_offset[r+1]

        row_nnz = 0
        for i in range(start, end):
            if mask[i]:
                row_nnz += 1

        nnz += row_nnz
        new_row_offset[r+1] = nnz

    return new_row_offset, new_col_idx, new_values


# Pratical 5 Features

In [ ]:
# Training using numba Features csr v1. vectorized
@njit
def get_row_from_csr(matrix_csr, row_idx):
    row_offset, col_indices, values = matrix_csr
    start = row_offset[row_idx]
    end = row_offset[row_idx + 1]
    return col_indices[start:end], values[start:end]

@njit
def get_nrows_csr(matrix_csr):
    row_offset, _, _ = matrix_csr
    return len(row_offset) - 1

@njit
def remove_element(x_list:np.ndarray, a:float)-> np.ndarray:
    keep_mask = x_list != a
    new_x_list = x_list[keep_mask]
    return new_x_list

@njit(fastmath=True)
def calc_metrics_numba_nb_csr_with_features(
        user_vectors:np.ndarray[np.ndarray],
        item_vectors:np.ndarray[np.ndarray],
        feature_vectors:np.ndarray[np.ndarray],
        user_biases:np.ndarray,
        item_biases:np.ndarray,
        ratings_by_movie_csr:tuple[np.ndarray, np.ndarray, np.ndarray],
        features_by_movie_csr:tuple[np.ndarray, np.ndarray, np.ndarray],
        movies_by_feature_csr:tuple[np.ndarray, np.ndarray, np.ndarray],
        precision:float,
        vector_penality:float,
        bias_penality:float,
        with_features=False):
    N = get_nrows_csr(ratings_by_movie_csr)
    K = user_vectors.shape[1]
    se = 0.0
    se_biases = 0.0
    cnt = 0
    vmean_vectors = np.empty((N,K))
    for n in range(N):
        users, ratings = get_row_from_csr(ratings_by_movie_csr, n)
        nusers = len(users)
        if nusers == 0:
            continue

        U = user_vectors[users]             # (nusers, K)
        v = item_vectors[n]                 # (K,)
        pred = U @ v + item_biases[n]  + user_biases[users] # shape (nusers,)
        err = ratings - pred

        se += np.sum(err ** 2)
        cnt += nusers

        features, _ = get_row_from_csr(features_by_movie_csr, n)
        Fn = len(features)
        vmean_vectors[n, :] = 1 / np.sqrt(Fn) * np.sum(feature_vectors[features, :], axis=0)

    rmse = np.sqrt(se / cnt)
    loss = (
        precision / 2.0 * se
        + bias_penality / 2.0 * (np.sum(user_biases ** 2) + np.sum(item_biases ** 2))
        + vector_penality / 2.0 * (
            + np.sum((item_vectors - with_features * vmean_vectors)** 2)
            + np.sum(user_vectors ** 2)
            + with_features * np.sum(vmean_vectors ** 2)
            )

    )
    return rmse, loss


@njit(parallel=True, fastmath=True)
def als_numba_iteration_nb_csr_with_features(
        user_vectors,
        item_vectors,
        feature_vectors,
        user_biases,
        item_biases,
        ratings_by_user_csr,
        ratings_by_item_csr,
        precision,
        vector_penality,
        bias_penality,
        K,
        with_features=False):
    M = get_nrows_csr(ratings_by_user_csr)
    N = get_nrows_csr(ratings_by_item_csr)
    F = get_nrows_csr(movies_by_feature_csr)
    def update_user(m: int):
        items, ratings = get_row_from_csr(ratings_by_user_csr, m)
        nitems = len(items)
        if nitems == 0:
            return user_biases[m], user_vectors[m]

        V = item_vectors[items]                 # (nitems, K)
        u = user_vectors[m]                     # (K,)
        item_bias = item_biases[items]          # (nitems,)
        pred = V @ u.T + item_bias
        resid = ratings - pred

        s = precision * np.sum(resid)
        user_bias = s / (precision * nitems + bias_penality)

        resid2 = ratings - user_bias - item_bias

        linsum_v = precision * (V.T @ resid2.T)
        transition_matrix = vector_penality * np.identity(K) + precision * (V.T @ V)

        user_vector = np.linalg.solve(transition_matrix, linsum_v)
        return user_bias, user_vector

    def update_item(n: int):
        users, ratings = get_row_from_csr(ratings_by_item_csr, n)
        nusers = len(users)
        if nusers == 0:
            return item_biases[n], item_vectors[n]

        U = user_vectors[users]                  # (nusers, K)
        v = item_vectors[n]                      # (K,)
        user_bias = user_biases[users]           # (nusers,)
        pred = U @ v.T + user_bias
        resid = ratings - pred

        s = precision * np.sum(resid)
        item_bias = s / (precision * nusers + bias_penality)

        resid2 = ratings - user_bias- item_bias

        linsum_u = precision * (U.T @ resid2.T)

        features, _ = get_row_from_csr(features_by_movie_csr, n)
        Fn = len(features)
        linsum_f = vector_penality / np.sqrt(Fn) * np.sum(feature_vectors[features, :], axis=0)
        transition_matrix = vector_penality * np.identity(K) + precision * (U.T @ U)

        item_vector = np.linalg.solve(transition_matrix, linsum_u + with_features * linsum_f)
        return item_bias, item_vector

    def update_feature(i: int):
        items, _ = get_row_from_csr(movies_by_feature_csr, i)
        nitems = len(items)
        Ci = 1
        vect_sum = np.zeros(K)
        item_vectors[items]
        for nidx in prange(len(items)):
            n = items[nidx]
            features, _ = get_row_from_csr(features_by_movie_csr, n)
            Fn = len(features)
            features_without_i = remove_element(features, i)
            features_without_i_vector = feature_vectors[features_without_i, :]
            mean_feature = 1 / Fn * np.sum(features_without_i_vector, axis=0)
            Ci += 1/ Fn
            vect_sum += item_vectors[n, :] / np.sqrt(Fn) - mean_feature

        feature_vector = 1/Ci * vect_sum
        return feature_vector

    for m in prange(M):
        user_biases[m], user_vectors[m, :] = update_user(m)

    for n in prange(N):
        item_biases[n], item_vectors[n, :] = update_item(n)

    for i in prange(F):
        feature_vectors[i, :] = update_feature(i)


@show_execution_duration(return_time=True)
def als_update_embeddings_and_biases_nb_csr_with_features(
    ratings_by_user_train_csr,
    ratings_by_movie_train_csr,
    ratings_by_user_test_csr,
    ratings_by_movie_test_csr,
    features_by_movie_csr,
    movies_by_feature_csr,
    precision,
    bias_penality,
    vector_penality,
    K=10,
    with_features=False,
    max_iter=10,
    beta=0.01,
    seed=None
):
    M = get_nrows_csr(ratings_by_user_train_csr)
    N = get_nrows_csr(ratings_by_movie_train_csr)
    F = get_nrows_csr(movies_by_feature_csr)
    # Initialize embeddings and biases
    np.random.seed(seed)


    # Initializing features
    fmean = np.zeros(K)
    fcov = beta * np.identity(K)
    feature_vectors = np.random.multivariate_normal(fmean, fcov, F)

    # Initializing users
    umean = np.zeros(K)
    ucov = (1.0 / np.sqrt(K)) * np.identity(K)
    user_vectors = np.random.multivariate_normal(umean, ucov, M)

    # Initializing items
    item_vectors = np.empty((N, K))
    for n in range(N):
        features, _ = get_row_from_csr(features_by_movie_csr, n)
        Fn = len(features)
        vmean = 1 / np.sqrt(Fn) * np.sum(feature_vectors[features, :], axis=0)
        if  np.sum(np.abs(vmean) > 100):
            print(vmean, np.abs(vmean) > 100)
        vcov = vector_penality * np.identity(K)
        item_vectors[n, :] = np.random.multivariate_normal(vmean, vcov)

    user_biases = np.zeros(M)
    item_biases = np.zeros(N)

    train_rmse_hist = []
    train_loss_hist = []
    test_rmse_hist = []
    test_loss_hist = []

    for it in tqdm(range(max_iter), unit="epoch"):
        als_numba_iteration_nb_csr_with_features(
            user_vectors,
            item_vectors,
            feature_vectors,
            user_biases,
            item_biases,
            ratings_by_user_train_csr,
            ratings_by_movie_train_csr,
            precision, vector_penality,
            bias_penality,
            K,
            with_features=with_features
        )

        train_rmse, train_loss = calc_metrics_numba_nb_csr_with_features(
            user_vectors,
            item_vectors,
            feature_vectors,
            user_biases,
            item_biases,
            ratings_by_movie_train_csr,
            features_by_movie_csr,
            movies_by_feature_csr,
            precision,
            vector_penality,
            bias_penality,
            with_features=with_features)
        test_rmse, test_loss = calc_metrics_numba_nb_csr_with_features(
            user_vectors,
            item_vectors,
            feature_vectors,
            user_biases,
            item_biases,
            ratings_by_movie_test_csr,
            features_by_movie_csr,
            movies_by_feature_csr,
            precision,
            vector_penality,
            bias_penality,
            with_features=with_features)

        train_rmse_hist.append(train_rmse)
        train_loss_hist.append(train_loss)
        test_rmse_hist.append(test_rmse)
        test_loss_hist.append(test_loss)

        if (it + 1) % 10 == 0 or it == 0:
            print("")
            print(f"epoch {it+1}/{max_iter} | train_loss={train_loss:.4f} "
                  f"test_loss={test_loss:.4f} | train_rmse={train_rmse:.4f} "
                  f"test_rmse={test_rmse:.4f}")
    return (user_vectors, user_biases, item_vectors, item_biases, feature_vectors,
            train_loss_hist, train_rmse_hist, test_loss_hist, test_rmse_hist
            )

In [ ]:
# precision = 10
# bias_penality = 0.1
# # vector_penality_range = [0.001, 0.01, 0.1, 1]
# vector_penality = 10
# K = 5
# max_iter = 5
# beta=0.01
# (
#     (user_vectors, user_biases, item_vectors, item_biases, feature_vectors,
#     train_loss_hist, train_rmse_hist, test_loss_hist, test_rmse_hist
#     ), training_starting_time, training_duration
# ) = als_update_embeddings_and_biases_nb_csr_with_features(ratings_by_user_train_csr,
#     ratings_by_movie_train_csr,
#     ratings_by_user_test_csr,
#     ratings_by_movie_test_csr,
#     features_by_movie_csr,
#     movies_by_feature_csr,
#     precision,
#     bias_penality,
#     vector_penality,
#     K,
#     with_features=True,
#     max_iter=max_iter,
#     beta=beta,
#     seed=None
# )

# plot_als_curves(train_loss_hist,
#         test_loss_hist,
#         train_rmse_hist,
#         test_rmse_hist,
#         K,
#         precision,
#         vector_penality,
#         bias_penality,
#         bias_only=False)

In [ ]:
@show_execution_duration("Big Loop")
def train_with_features(K_range, vector_penality_range, save=False):
    bias_only = False
    with_embeddings_path = model_folder /"features" / data_size_str/ "with_embeddings"
    bias_only_path = model_folder / "features" /data_size_str / "bias_only"
    for K in K_range:
        # for vector_penality in vector_penality_range:
        user_vectors = []
        item_vectors = []
        def fun_save():
            save_relative_path = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-lambda{precision}-tau{vector_penality}.joblib"
            save_relative_path_latest = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-latest.joblib"

            save_relative_path = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-lambda{precision}-tau{vector_penality}.joblib"
            save_relative_path_latest = f"{data_size_str}-als-K{K:02d}-iter{max_iter}-latest.joblib"
            save_path =  with_embeddings_path / save_relative_path
            save_path_latest = with_embeddings_path / save_relative_path_latest
            suffix = f"iter{max_iter}-with_embeddings"

            log_file = PROJECT_PATH / f"models/als_runs_{suffix}.yaml"
            log_file = log_file.as_posix()  # single YAML log for all joblib models
            data_model = {
                "user_vectors":user_vectors,
                "item_vectors":item_vectors,
                "feature_vectors":feature_vectors,
                "user_biases":user_biases,
                "item_biases":item_biases,
                "precision":precision,
                "bias_penality":bias_penality,
                "vector_penality":vector_penality,
                "K":K,
                "bias_only":bias_only,
                "train_loss_hist":train_loss_hist,
                "train_rmse_hist":train_rmse_hist,
                "test_loss_hist":test_loss_hist,
                "test_rmse_hist":test_rmse_hist,
                "max_iter":max_iter,
                "training_starting_time":training_starting_time,
                "training_duration": training_duration,
                "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            }
            args =  {
                "precision":precision,
                "bias_penality":bias_penality,
                "vector_penality":vector_penality,
                "K":K,
                "bias_only":bias_only,
                # "train_loss_hist":train_loss_hist,
                "train_rmse_hist":train_rmse_hist[-2:],
                # "test_loss_hist":test_loss_hist,
                "test_rmse_hist":test_rmse_hist[-2:],
                "max_iter":max_iter,
                "timestamp": datetime.now().strftime("%Y-%m-%d_%H-%M-%S"),
                "model_path":save_relative_path

            }

            # Dump with timestamp
            saved_path = dump_objects(
                data_model,
                path= save_path,
                timestamp=True
            )

            run_id, log_file = log_als_run_to_yaml(log_file, args)

            print("Logged run:", run_id)
            print("YAML log file:", log_file)

            saved_path_latest = dump_objects(
                data_model,
                path= save_path_latest,
                timestamp=False
            )

# ------------------------------------------------------------------------------

        for vector_penality in vector_penality_range:
            print(f"K:{K}, tau:{vector_penality}")
            (
                (user_vectors, user_biases, item_vectors, item_biases, feature_vectors,
                train_loss_hist, train_rmse_hist, test_loss_hist, test_rmse_hist
                ), training_starting_time, training_duration
            ) = als_update_embeddings_and_biases_nb_csr_with_features(ratings_by_user_train_csr,
                ratings_by_movie_train_csr,
                ratings_by_user_test_csr,
                ratings_by_movie_test_csr,
                features_by_movie_csr,
                movies_by_feature_csr,
                precision,
                bias_penality,
                vector_penality,
                K,
                with_features=True,
                max_iter=max_iter,
                beta=beta,
                seed=None
            )


            plot_als_curves(train_loss_hist,
                    test_loss_hist,
                    train_rmse_hist,
                    test_rmse_hist,
                    K,
                    precision,
                    vector_penality,
                    bias_penality,
                    bias_only=bias_only)
            if save:
                fun_save()
    return (user_vectors, user_biases, item_vectors, item_biases, feature_vectors,
                train_loss_hist, train_rmse_hist, test_loss_hist, test_rmse_hist
            )

In [ ]:
precision = 1
bias_penality = 1e-3

vector_penality_range = [0.1, 1, 10]
vector_penality_range = [1]

beta = 0.01
max_iter = 50
K_range=[8, 16]
_ = train_with_features(K_range, vector_penality_range, save=True)

In [ ]:
feat = _[4]

In [ ]:
def plot_vectors_umap(vectors, labels=None, n_neighbors=15, min_dist=0.1):
    reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist)
    points = reducer.fit_transform(vectors)

    plt.figure(figsize=(8, 6))
    plt.scatter(points[:, 0], points[:, 1])

    if labels is not None:
        for (x, y), label in zip(points, labels):
            plt.text(x, y, str(label), fontsize=9)

    plt.title("UMAP projection")
    plt.grid(True)
    plt.show()

In [ ]:
plot_vectors_umap(feat, labels=feature_id_to_idx.keys())

Load Model

In [ ]:
fitted_model['bias_penality']

In [ ]:
K = 16
models_path = model_folder / "features"/"ml-32m" / "with_embeddings"
exclude_pattern = "latest.joblib$"
models_K =  load_model_given_K(models_path, K, exclude_pattern=exclude_pattern)
fitted_model = models_K[-1]

In [ ]:
plot_als_curves_multiple_models(models_K)

In [ ]:
K = 16
models_path = model_folder / "ml-32m" / "with_embeddings"
exclude_pattern = "latest.joblib$"
models_K =  load_model_given_K(models_path, K, exclude_pattern=exclude_pattern)
fitted_model = models_K[-1]

In [ ]:
plot_als_curves_multiple_models(models_K)

In [ ]:
title = "Spider.*Man"
ids, titles = find_movie_id_by_title(title, movies_title, movie_idx_to_id, movie_id_to_idx)
results = get_movie_info_from_id(
    ids,
    movie_id_to_idx,
    feature_idx_to_id,
    movies_title,
    features_by_movie_csr
    )
pd.DataFrame(results)

In [ ]:
movies_title[movie_id_to_idx[ids[0]]]

In [ ]:
# Toy Story
dummy_ids = [120474]

# John Wick
# dummy_ids = [115149]

# Spider-Man (2002)
# dummy_ids = [5349, 8636, 52722]

rated_movie_info = get_movie_info_from_id(
    dummy_ids, movie_id_to_idx, feature_idx_to_id,
    movies_title, features_by_movie_csr)
pd.DataFrame(rated_movie_info)

In [ ]:
dummy_idx = get_idx_from_id(dummy_ids, movie_id_to_idx)
dummy_ratings_by_user = [[(idx, 5) for idx in dummy_idx],]

bias_factor = 0.0005
max_iter = 100
recommended_movies_ids, recommended_movie_score  = get_top_recommended_item(
        dummy_ratings_by_user,
        movie_idx_to_id,
        fitted_model['item_vectors'],
        fitted_model['item_biases'],
        fitted_model['precision'],
        fitted_model['bias_penality'],
        fitted_model['vector_penality'],
        max_iter=max_iter,
        ntop=20,
        bias_factor=bias_factor,
    )

recommended_info = []
for user_recommended_movie_id in recommended_movies_ids:
    recommended_movie_info = get_movie_info_from_id(
    user_recommended_movie_id, movie_id_to_idx,
    feature_idx_to_id,
    movies_title,
    features_by_movie_csr)

pd.DataFrame(recommended_movie_info)